In [10]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"D:\GeneVISTA")
CLINVAR_FOLDER = PROJECT_ROOT / "data" / "raw" / "clinvar"

clinvar_files = sorted(CLINVAR_FOLDER.glob("variant_summary_*.txt.gz"))

print("ClinVar files found:", len(clinvar_files))

ClinVar files found: 6


In [11]:
columns_by_file = {}

for file in clinvar_files:
    header = pd.read_csv(
        file,
        sep="\t",
        compression="gzip",
        nrows=0
    )

    columns_by_file[file.name] = list(header.columns)

    print(f"\n{file.name}")
    print(f"Columns: {len(header.columns)}")
    print(header.columns.tolist())


variant_summary_2022-01.txt.gz
Columns: 34
['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)', 'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList', 'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession', 'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele', 'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 'Guidelines', 'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF']

variant_summary_2023-01.txt.gz
Columns: 34
['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)', 'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList', 'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession', 'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele', 'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 

In [12]:
common_columns = set(columns_by_file[clinvar_files[0].name])

for file in clinvar_files[1:]:
    common_columns &= set(columns_by_file[file.name])

common_columns = sorted(common_columns)

print("Common columns in all six files:", len(common_columns))
print(common_columns)

Common columns in all six files: 34
['#AlleleID', 'AlternateAllele', 'AlternateAlleleVCF', 'Assembly', 'Chromosome', 'ChromosomeAccession', 'ClinSigSimple', 'ClinicalSignificance', 'Cytogenetic', 'GeneID', 'GeneSymbol', 'Guidelines', 'HGNC_ID', 'LastEvaluated', 'Name', 'NumberSubmitters', 'Origin', 'OriginSimple', 'OtherIDs', 'PhenotypeIDS', 'PhenotypeList', 'PositionVCF', 'RCVaccession', 'RS# (dbSNP)', 'ReferenceAllele', 'ReferenceAlleleVCF', 'ReviewStatus', 'Start', 'Stop', 'SubmitterCategories', 'TestedInGTR', 'Type', 'VariationID', 'nsv/esv (dbVar)']


In [13]:
CORE_COLUMNS = [
    "VariationID",
    "#AlleleID",
    "Type",
    "Name",
    "GeneID",
    "GeneSymbol",
    "HGNC_ID",
    "ClinicalSignificance",
    "ClinSigSimple",
    "LastEvaluated",
    "PhenotypeIDS",
    "PhenotypeList",
    "Origin",
    "OriginSimple",
    "Assembly",
    "Chromosome",
    "Start",
    "Stop",
    "ReferenceAllele",
    "AlternateAllele",
    "ReviewStatus",
    "NumberSubmitters",
    "SubmitterCategories",
    "RCVaccession"
]

print("Core columns selected:", len(CORE_COLUMNS))

Core columns selected: 24


In [14]:
sample_2022 = pd.read_csv(
    clinvar_files[0],
    sep="\t",
    compression="gzip",
    usecols=CORE_COLUMNS,
    dtype=str,
    nrows=10_000
)

print("Sample shape:", sample_2022.shape)

print("\nClinical significance counts:")
print(sample_2022["ClinicalSignificance"].value_counts().head(15))

print("\nAssembly counts:")
print(sample_2022["Assembly"].value_counts(dropna=False))

print("\nMissing values in important columns:")
print(
    sample_2022[
        ["VariationID", "GeneSymbol", "ClinicalSignificance", "ReviewStatus"]
    ]
    .isna()
    .sum()
)

Sample shape: (10000, 24)

Clinical significance counts:
ClinicalSignificance
Pathogenic                                             7105
Pathogenic/Likely pathogenic                            894
Conflicting interpretations of pathogenicity            576
Likely pathogenic                                       410
Uncertain significance                                  409
Benign                                                  173
risk factor                                             141
Benign/Likely benign                                     44
Likely benign                                            36
Affects                                                  35
association                                              34
drug response                                            26
Pathogenic, risk factor                                  24
no interpretation for the single variant                 24
Conflicting interpretations of pathogenicity, other      16
Name: count, dtype: in

In [15]:
sample_2022 = pd.read_csv(
    clinvar_files[0],
    sep="\t",
    compression="gzip",
    usecols=CORE_COLUMNS,
    dtype=str,
    nrows=10_000
)

print("Sample shape:", sample_2022.shape)

print("\nClinical significance counts:")
print(sample_2022["ClinicalSignificance"].value_counts().head(15))

print("\nAssembly counts:")
print(sample_2022["Assembly"].value_counts(dropna=False))

print("\nMissing values in important columns:")
print(
    sample_2022[
        ["VariationID", "GeneSymbol", "ClinicalSignificance", "ReviewStatus"]
    ]
    .isna()
    .sum()
)

Sample shape: (10000, 24)

Clinical significance counts:
ClinicalSignificance
Pathogenic                                             7105
Pathogenic/Likely pathogenic                            894
Conflicting interpretations of pathogenicity            576
Likely pathogenic                                       410
Uncertain significance                                  409
Benign                                                  173
risk factor                                             141
Benign/Likely benign                                     44
Likely benign                                            36
Affects                                                  35
association                                              34
drug response                                            26
Pathogenic, risk factor                                  24
no interpretation for the single variant                 24
Conflicting interpretations of pathogenicity, other      16
Name: count, dtype: in

In [18]:
from collections import Counter

input_file = CLINVAR_FOLDER / "variant_summary_2022-01.txt.gz"

processed_folder = PROJECT_ROOT / "data" / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)

output_file = processed_folder / "clinvar_clean_2022-01.csv.gz"
temporary_file = processed_folder / "clinvar_clean_2022-01.partial.csv.gz"

required_columns = [
    "VariationID",
    "GeneSymbol",
    "ClinicalSignificance",
    "ReviewStatus"
]

# Treat these placeholders as missing values.
missing_values = ["", "-", ".", "na", "n/a", "nan", "none", "null"]

clinical_labels = {
    "benign": "benign",
    "likely benign": "benign",
    "benign/likely benign": "benign",
    "uncertain significance": "vus",
    "likely pathogenic": "pathogenic",
    "pathogenic": "pathogenic",
    "pathogenic/likely pathogenic": "pathogenic"
}


def simplify_clinical_state(value):
    label = value.strip().lower()

    if "conflicting" in label:
        return "conflicting"

    # Keep uncommon or mixed classifications in a separate group.
    return clinical_labels.get(label, "other")

In [19]:
if not input_file.exists():
    raise FileNotFoundError(f"Input file not found: {input_file}")

if output_file.exists():
    raise FileExistsError(f"A cleaned file already exists: {output_file}")

total_rows = 0
assembly_rows_removed = 0
missing_rows_removed = 0
saved_rows = 0
state_counts = Counter()

chunks = pd.read_csv(
    input_file,
    sep="\t",
    compression="gzip",
    usecols=CORE_COLUMNS,
    dtype="string",
    chunksize=100_000
)

# Opening once also writes the CSV header only once.
import gzip

with gzip.open(temporary_file, "wt", encoding="utf-8", newline="") as saved_file:
    final_columns = CORE_COLUMNS + ["snapshot", "clinical_state"]
    pd.DataFrame(columns=final_columns).to_csv(saved_file, index=False)

    for chunk_number, chunk in enumerate(chunks, start=1):
        total_rows += len(chunk)

        # Use the same genome assembly across all snapshots.
        keep_assembly = chunk["Assembly"].str.strip().eq("GRCh38").fillna(False)
        assembly_rows_removed += int((~keep_assembly).sum())
        clean_chunk = chunk.loc[keep_assembly, CORE_COLUMNS].copy()

        # Remove spaces and standardize missing values in required fields.
        for column in required_columns:
            values = clean_chunk[column].str.strip()
            clean_chunk[column] = values.mask(
                values.str.lower().isin(missing_values)
            )

        rows_before = len(clean_chunk)
        clean_chunk = clean_chunk.dropna(subset=required_columns)
        missing_rows_removed += rows_before - len(clean_chunk)

        clean_chunk["snapshot"] = "2022-01"
        clean_chunk["clinical_state"] = (
            clean_chunk["ClinicalSignificance"].map(simplify_clinical_state)
        )

        clean_chunk.to_csv(saved_file, index=False, header=False)

        saved_rows += len(clean_chunk)
        state_counts.update(clean_chunk["clinical_state"])

        print(f"Chunk {chunk_number}: {len(clean_chunk):,} rows saved")

assert total_rows == assembly_rows_removed + missing_rows_removed + saved_rows
assert saved_rows > 0, "No rows were retained. Check the input before continuing."

temporary_file.replace(output_file)

print("\n2022 CLEANING SUMMARY")
print(f"Original rows:                  {total_rows:,}")
print(f"Removed — other assemblies:    {assembly_rows_removed:,}")
print(f"Removed — missing fields:      {missing_rows_removed:,}")
print(f"Saved rows:                    {saved_rows:,}")

print("\nClinical states:")
for state, count in sorted(state_counts.items()):
    print(f"  {state}: {count:,}")

print(f"\nSaved to: {output_file}")

Chunk 1: 48,740 rows saved
Chunk 2: 48,852 rows saved
Chunk 3: 49,508 rows saved
Chunk 4: 49,324 rows saved
Chunk 5: 50,002 rows saved
Chunk 6: 45,855 rows saved
Chunk 7: 48,620 rows saved
Chunk 8: 49,855 rows saved
Chunk 9: 48,552 rows saved
Chunk 10: 40,708 rows saved
Chunk 11: 47,685 rows saved
Chunk 12: 49,984 rows saved
Chunk 13: 49,987 rows saved
Chunk 14: 47,143 rows saved
Chunk 15: 49,871 rows saved
Chunk 16: 49,895 rows saved
Chunk 17: 48,520 rows saved
Chunk 18: 49,555 rows saved
Chunk 19: 48,946 rows saved
Chunk 20: 49,959 rows saved
Chunk 21: 49,876 rows saved
Chunk 22: 49,976 rows saved
Chunk 23: 49,962 rows saved
Chunk 24: 5,609 rows saved

2022 CLEANING SUMMARY
Original rows:                  2,315,318
Removed — other assemblies:    1,187,052
Removed — missing fields:      1,282
Saved rows:                    1,126,984

Clinical states:
  benign: 477,745
  conflicting: 53,948
  other: 15,751
  pathogenic: 158,308
  vus: 421,232

Saved to: D:\GeneVISTA\data\processed\clin

In [20]:
cleaned_file = processed_folder / "clinvar_clean_2022-01.csv.gz"

checked_rows = 0
checked_states = Counter()
seen_variant_ids = set()
repeated_variant_rows = 0
other_labels = Counter()

expected_columns = CORE_COLUMNS + ["snapshot", "clinical_state"]
allowed_states = {"benign", "pathogenic", "vus", "conflicting", "other"}

for chunk in pd.read_csv(
    cleaned_file,
    compression="gzip",
    dtype="string",
    chunksize=100_000
):
    # Check the saved columns and snapshot information.
    assert list(chunk.columns) == expected_columns, "Unexpected columns found."
    assert chunk["Assembly"].str.strip().eq("GRCh38").all(), "Other assembly found."
    assert chunk["snapshot"].eq("2022-01").all(), "Wrong snapshot value found."

    # Required fields should contain real values.
    for column in required_columns:
        values = chunk[column].str.strip()

        assert values.notna().all(), f"Missing values in {column}"
        assert not values.str.lower().isin(missing_values).any(), (
            f"Missing-value placeholders in {column}"
        )

    assert chunk["clinical_state"].isin(allowed_states).all(), (
        "Unexpected clinical state found."
    )

    expected_states = chunk["ClinicalSignificance"].map(simplify_clinical_state)
    assert chunk["clinical_state"].eq(expected_states).all(), (
        "Clinical state does not match the original classification."
    )

    # Count repeated IDs across the entire file, including chunk boundaries.
    for variant_id in chunk["VariationID"]:
        if variant_id in seen_variant_ids:
            repeated_variant_rows += 1
        else:
            seen_variant_ids.add(variant_id)

    checked_rows += len(chunk)
    checked_states.update(chunk["clinical_state"])

    other_labels.update(
        chunk.loc[
            chunk["clinical_state"].eq("other"),
            "ClinicalSignificance"
        ]
    )

assert checked_rows == 1_126_984, "Row count differs from the cleaning summary."

expected_counts = {
    "benign": 477_745,
    "conflicting": 53_948,
    "other": 15_751,
    "pathogenic": 158_308,
    "vus": 421_232
}
assert dict(checked_states) == expected_counts, "Clinical state counts differ."

print("2022 VALIDATION PASSED")
print(f"Rows checked: {checked_rows:,}")
print(f"Unique VariationIDs: {len(seen_variant_ids):,}")
print(f"Rows repeating a VariationID: {repeated_variant_rows:,}")

print("\nMost common classifications grouped as 'other':")
for label, count in other_labels.most_common(10):
    print(f"  {label}: {count:,}")

2022 VALIDATION PASSED
Rows checked: 1,126,984
Unique VariationIDs: 1,126,575
Rows repeating a VariationID: 409

Most common classifications grouped as 'other':
  not provided: 10,446
  drug response: 1,893
  other: 1,528
  no interpretation for the single variant: 635
  risk factor: 437
  association: 292
  Affects: 147
  Pathogenic, other: 93
  Pathogenic, risk factor: 39
  protective: 32


In [21]:
import gzip


def clean_snapshot(source_file):
    snapshot = source_file.name.removeprefix("variant_summary_").removesuffix(".txt.gz")
    destination = processed_folder / f"clinvar_clean_{snapshot}.csv.gz"
    temporary = processed_folder / f"clinvar_clean_{snapshot}.partial.csv.gz"

    # Protect files from previous runs.
    if destination.exists():
        print(f"{snapshot}: already exists, skipped.")
        return None

    print(f"\nCleaning {snapshot}...")

    rows_read = 0
    assembly_removed = 0
    missing_removed = 0
    rows_saved = 0
    clinical_counts = Counter()
    columns_to_save = CORE_COLUMNS + ["snapshot", "clinical_state"]

    chunks = pd.read_csv(
        source_file,
        sep="\t",
        compression="gzip",
        usecols=CORE_COLUMNS,
        dtype="string",
        chunksize=100_000
    )

    with gzip.open(temporary, "wt", encoding="utf-8", newline="") as handle:
        pd.DataFrame(columns=columns_to_save).to_csv(handle, index=False)

        for chunk_number, chunk in enumerate(chunks, start=1):
            rows_read += len(chunk)

            keep_rows = chunk["Assembly"].str.strip().eq("GRCh38").fillna(False)
            assembly_removed += int((~keep_rows).sum())
            cleaned = chunk.loc[keep_rows, CORE_COLUMNS].copy()

            for column in required_columns:
                values = cleaned[column].str.strip()
                cleaned[column] = values.mask(
                    values.str.lower().isin(missing_values)
                )

            before_removing_missing = len(cleaned)
            cleaned = cleaned.dropna(subset=required_columns)
            missing_removed += before_removing_missing - len(cleaned)

            cleaned["snapshot"] = snapshot
            cleaned["clinical_state"] = (
                cleaned["ClinicalSignificance"].map(simplify_clinical_state)
            )

            cleaned.to_csv(handle, index=False, header=False)

            rows_saved += len(cleaned)
            clinical_counts.update(cleaned["clinical_state"])

            if chunk_number % 5 == 0:
                print(f"  Read {rows_read:,} rows; kept {rows_saved:,}")

    assert rows_saved > 0, f"{snapshot}: no rows retained."
    assert rows_read == assembly_removed + missing_removed + rows_saved

    # Read the complete temporary file before giving it its final name.
    print("  Checking the saved data...")

    checked_rows = 0
    checked_counts = Counter()

    for chunk in pd.read_csv(temporary, dtype="string", chunksize=100_000):
        assert list(chunk.columns) == columns_to_save
        assert chunk["Assembly"].str.strip().eq("GRCh38").all()
        assert chunk["snapshot"].eq(snapshot).all()

        for column in required_columns:
            values = chunk[column].str.strip()
            assert values.notna().all(), f"Missing values in {column}"
            assert not values.str.lower().isin(missing_values).any()

        expected_states = chunk["ClinicalSignificance"].map(simplify_clinical_state)
        assert chunk["clinical_state"].eq(expected_states).all()

        checked_rows += len(chunk)
        checked_counts.update(chunk["clinical_state"])

    assert checked_rows == rows_saved, "Saved row count does not match."
    assert checked_counts == clinical_counts, "Clinical state counts do not match."

    temporary.replace(destination)
    print(f"  {snapshot}: validation passed — {rows_saved:,} rows saved.")

    return {
        "snapshot": snapshot,
        "original_rows": rows_read,
        "other_assembly_rows": assembly_removed,
        "missing_field_rows": missing_removed,
        "saved_rows": rows_saved,
        **{state: clinical_counts[state] for state in [
            "benign", "pathogenic", "vus", "conflicting", "other"
        ]}
    }

In [22]:
remaining_snapshots = [
    "2023-01",
    "2024-01",
    "2025-01",
    "2026-01",
    "2026-09"
]

remaining_files = [
    CLINVAR_FOLDER / f"variant_summary_{snapshot}.txt.gz"
    for snapshot in remaining_snapshots
]

# Confirm all inputs exist before starting.
for source_file in remaining_files:
    if not source_file.exists():
        raise FileNotFoundError(f"Snapshot not found: {source_file}")

cleaning_results = []

for source_file in remaining_files:
    result = clean_snapshot(source_file)

    if result is not None:
        cleaning_results.append(result)

if cleaning_results:
    cleaning_summary = pd.DataFrame(cleaning_results)
    display(cleaning_summary)
else:
    print("All five output files already exist; no files were processed.")


Cleaning 2023-01...
  Read 500,000 rows; kept 247,177
  Read 1,000,000 rows; kept 481,598
  Read 1,500,000 rows; kept 726,303
  Read 2,000,000 rows; kept 973,201
  Read 2,500,000 rows; kept 1,221,283
  Read 3,000,000 rows; kept 1,469,128
  Checking the saved data...
  2023-01: validation passed — 1,595,467 rows saved.

Cleaning 2024-01...
  Read 500,000 rows; kept 247,539
  Read 1,000,000 rows; kept 481,961
  Read 1,500,000 rows; kept 726,765
  Read 2,000,000 rows; kept 973,986
  Read 2,500,000 rows; kept 1,222,156
  Read 3,000,000 rows; kept 1,470,005
  Read 3,500,000 rows; kept 1,715,814
  Read 4,000,000 rows; kept 1,965,783
  Read 4,500,000 rows; kept 2,215,542
  Checking the saved data...
  2024-01: validation passed — 2,365,518 rows saved.

Cleaning 2025-01...
  Read 500,000 rows; kept 247,565
  Read 1,000,000 rows; kept 490,824
  Read 1,500,000 rows; kept 735,581
  Read 2,000,000 rows; kept 982,764
  Read 2,500,000 rows; kept 1,230,928
  Read 3,000,000 rows; kept 1,478,772
  Rea

,snapshot,original_rows,other_assembly_rows,missing_field_rows,saved_rows,benign,pathogenic,vus,conflicting,other
0,2023-01,3255919,1659182,1270,1595467,660971,201758,642050,73918,16770
1,2024-01,4800563,2433913,1132,2365518,862292,240727,1138920,107283,16296
2,2025-01,6181555,3122061,1618,3057876,1147412,288849,1475455,129378,16782
3,2026-01,8378417,4221448,246825,3910144,1283917,328630,2129054,155386,13157
4,2026-09,9049133,4558256,246586,4244291,1373925,355061,2336246,166047,13012


In [23]:
snapshots_to_check = ["2025-01", "2026-01", "2026-09"]
missing_field_report = []
missing_pattern_report = []

for snapshot in snapshots_to_check:
    source_file = CLINVAR_FOLDER / f"variant_summary_{snapshot}.txt.gz"

    field_counts = Counter()
    pattern_counts = Counter()
    assembly_rows = 0
    removed_rows = 0

    print(f"Checking missing fields in {snapshot}...")

    chunks = pd.read_csv(
        source_file,
        sep="\t",
        compression="gzip",
        usecols=["Assembly"] + required_columns,
        dtype="string",
        chunksize=100_000
    )

    for chunk in chunks:
        keep_rows = chunk["Assembly"].str.strip().eq("GRCh38").fillna(False)
        chunk = chunk.loc[keep_rows].copy()
        assembly_rows += len(chunk)

        missing_flags = pd.DataFrame(index=chunk.index)

        for column in required_columns:
            values = chunk[column].str.strip()

            missing_flags[column] = (
                values.isna()
                | values.str.lower().isin(missing_values)
            )

            field_counts[column] += int(missing_flags[column].sum())

        missing_any = missing_flags.any(axis=1)
        removed_rows += int(missing_any.sum())

        # Count combinations too: one row may have several missing fields.
        combinations = missing_flags.loc[missing_any].value_counts()

        for flags, count in combinations.items():
            missing_names = [
                column
                for column, is_missing in zip(required_columns, flags)
                if is_missing
            ]
            pattern_counts[" + ".join(missing_names)] += int(count)

    missing_field_report.append({
        "snapshot": snapshot,
        "GRCh38_rows": assembly_rows,
        "removed_rows": removed_rows,
        "removed_percent": round(
            100 * removed_rows / assembly_rows, 2
        ) if assembly_rows else 0,
        **{column: field_counts[column] for column in required_columns}
    })

    for pattern, count in pattern_counts.most_common():
        missing_pattern_report.append({
            "snapshot": snapshot,
            "missing_fields": pattern,
            "rows": count
        })

print("\nMissing fields by snapshot:")
display(pd.DataFrame(missing_field_report))

print("\nCombinations of missing fields:")
display(pd.DataFrame(missing_pattern_report))

Checking missing fields in 2025-01...
Checking missing fields in 2026-01...
Checking missing fields in 2026-09...

Missing fields by snapshot:


,snapshot,GRCh38_rows,removed_rows,removed_percent,VariationID,GeneSymbol,ClinicalSignificance,ReviewStatus
0,2025-01,3059494,1618,0.05,0,1240,379,379
1,2026-01,4156969,246825,5.94,0,1437,245395,245395
2,2026-09,4490877,246586,5.49,0,1561,245032,245032



Combinations of missing fields:


,snapshot,missing_fields,rows
0,2025-01,GeneSymbol,1239
1,2025-01,ClinicalSignificance + ReviewStatus,378
2,2025-01,GeneSymbol + ClinicalSignificance + ReviewStatus,1
3,2026-01,ClinicalSignificance + ReviewStatus,245388
4,2026-01,GeneSymbol,1430
5,2026-01,GeneSymbol + ClinicalSignificance + ReviewStatus,7
6,2026-09,ClinicalSignificance + ReviewStatus,245025
7,2026-09,GeneSymbol,1554
8,2026-09,GeneSymbol + ClinicalSignificance + ReviewStatus,7


In [24]:
snapshots_to_check = ["2025-01", "2026-01", "2026-09"]

classification_columns = [
    "ClinicalSignificance",
    "ReviewStatus",
    "SomaticClinicalImpact",
    "Oncogenicity"
]

classification_report = []
classification_examples = []


def has_value(series):
    values = series.str.strip()
    return values.notna() & ~values.str.lower().isin(missing_values)


for snapshot in snapshots_to_check:
    source_file = CLINVAR_FOLDER / f"variant_summary_{snapshot}.txt.gz"
    evidence_counts = Counter()
    examples_collected = 0

    print(f"Checking classifications in {snapshot}...")

    chunks = pd.read_csv(
        source_file,
        sep="\t",
        compression="gzip",
        usecols=["VariationID", "Assembly"] + classification_columns,
        dtype="string",
        chunksize=100_000
    )

    for chunk in chunks:
        is_grch38 = (
            chunk["Assembly"].str.strip().eq("GRCh38").fillna(False)
        )

        missing_both = (
            ~has_value(chunk["ClinicalSignificance"])
            & ~has_value(chunk["ReviewStatus"])
        )

        excluded = chunk.loc[is_grch38 & missing_both].copy()

        if excluded.empty:
            continue

        has_somatic = has_value(excluded["SomaticClinicalImpact"])
        has_oncogenicity = has_value(excluded["Oncogenicity"])

        evidence_counts["missing_both"] += len(excluded)
        evidence_counts["somatic_only"] += int(
            (has_somatic & ~has_oncogenicity).sum()
        )
        evidence_counts["oncogenicity_only"] += int(
            (~has_somatic & has_oncogenicity).sum()
        )
        evidence_counts["both_present"] += int(
            (has_somatic & has_oncogenicity).sum()
        )
        evidence_counts["neither_present"] += int(
            (~has_somatic & ~has_oncogenicity).sum()
        )

        if examples_collected < 5:
            examples = excluded[
                ["VariationID"] + classification_columns
            ].head(5 - examples_collected).copy()

            examples.insert(0, "snapshot", snapshot)
            classification_examples.append(examples)
            examples_collected += len(examples)

    classification_report.append({
        "snapshot": snapshot,
        **{
            name: evidence_counts[name]
            for name in [
                "missing_both",
                "somatic_only",
                "oncogenicity_only",
                "both_present",
                "neither_present"
            ]
        }
    })

print("\nClassification coverage for excluded records:")
display(pd.DataFrame(classification_report))

if classification_examples:
    print("\nExample records:")
    display(pd.concat(classification_examples, ignore_index=True))

Checking classifications in 2025-01...
Checking classifications in 2026-01...
Checking classifications in 2026-09...

Classification coverage for excluded records:


,snapshot,missing_both,somatic_only,oncogenicity_only,both_present,neither_present
0,2025-01,379,12,367,0,0
1,2026-01,245395,556,654,17,244168
2,2026-09,245032,625,957,23,243427



Example records:


,snapshot,VariationID,ClinicalSignificance,ReviewStatus,SomaticClinicalImpact,Oncogenicity
0,2025-01,375889,-,-,-,Oncogenic
1,2025-01,375902,-,-,-,Oncogenic
2,2025-01,375908,-,-,-,Likely oncogenic
3,2025-01,375916,-,-,-,Likely oncogenic
4,2025-01,375917,-,-,-,Uncertain significance
5,2026-01,4507303,-,-,-,-
6,2026-01,4463740,-,-,-,-
7,2026-01,4463746,-,-,-,-
8,2026-01,4463749,-,-,-,-
9,2026-01,4463758,-,-,-,-


In [25]:
report = pd.DataFrame(classification_report)

print("CLASSIFICATION COVERAGE")
print(report.to_string(index=False))

for row in classification_report:
    counted_rows = (
        row["somatic_only"]
        + row["oncogenicity_only"]
        + row["both_present"]
        + row["neither_present"]
    )

    assert counted_rows == row["missing_both"], (
        f"Counts do not match for {row['snapshot']}"
    )

print("\nAll classification counts match.")

CLASSIFICATION COVERAGE
snapshot  missing_both  somatic_only  oncogenicity_only  both_present  neither_present
 2025-01           379            12                367             0                0
 2026-01        245395           556                654            17           244168
 2026-09        245032           625                957            23           243427

All classification counts match.


In [26]:
import gzip

snapshots = [
    "2022-01", "2023-01", "2024-01",
    "2025-01", "2026-01", "2026-09"
]

presence_summary = []

for snapshot in snapshots:
    source_file = CLINVAR_FOLDER / f"variant_summary_{snapshot}.txt.gz"
    presence_file = processed_folder / f"clinvar_presence_{snapshot}.csv.gz"
    temporary = processed_folder / f"clinvar_presence_{snapshot}.partial.csv.gz"

    if presence_file.exists():
        print(f"{snapshot}: presence file already exists, skipped.")
        continue

    print(f"Recording presence for {snapshot}...")

    total_rows = 0
    missing_id_rows = 0
    status_counts = Counter()

    chunks = pd.read_csv(
        source_file,
        sep="\t",
        compression="gzip",
        usecols=["Assembly"] + required_columns,
        dtype="string",
        chunksize=100_000
    )

    columns = ["VariationID", "snapshot", "included_in_cleaned", "missing_fields"]

    with gzip.open(temporary, "wt", encoding="utf-8", newline="") as handle:
        pd.DataFrame(columns=columns).to_csv(handle, index=False)

        for chunk in chunks:
            keep_rows = chunk["Assembly"].str.strip().eq("GRCh38").fillna(False)
            chunk = chunk.loc[keep_rows].copy()

            missing = pd.DataFrame(index=chunk.index)

            for column in required_columns:
                values = chunk[column].str.strip()
                missing[column] = (
                    values.isna()
                    | values.str.lower().isin(missing_values)
                )
                chunk[column] = values

            missing_id_rows += int(missing["VariationID"].sum())

            # A row without an ID cannot be linked to a variant timeline.
            valid_id = ~missing["VariationID"]
            chunk = chunk.loc[valid_id]
            missing = missing.loc[valid_id]

            reasons = pd.Series("", index=chunk.index, dtype="string")

            for column in required_columns:
                reasons = reasons + missing[column].map({
                    True: column + ";",
                    False: ""
                })

            reasons = reasons.str.rstrip(";").replace("", "none")

            presence = pd.DataFrame({
                "VariationID": chunk["VariationID"],
                "snapshot": snapshot,
                "included_in_cleaned": ~missing.any(axis=1),
                "missing_fields": reasons
            })

            presence.to_csv(handle, index=False, header=False)

            total_rows += len(presence)
            status_counts.update(presence["included_in_cleaned"])

    # Verify that the compressed file can be read completely.
    checked_rows = 0
    checked_included = 0

    for chunk in pd.read_csv(temporary, dtype="string", chunksize=100_000):
        assert list(chunk.columns) == columns
        assert chunk["VariationID"].notna().all()
        assert chunk["snapshot"].eq(snapshot).all()
        assert chunk["included_in_cleaned"].isin(["True", "False"]).all()

        checked_rows += len(chunk)
        checked_included += int(chunk["included_in_cleaned"].eq("True").sum())

    assert checked_rows == total_rows
    assert checked_included == status_counts[True]

    # The inclusion count must match the existing cleaned dataset.
    cleaned_file = processed_folder / f"clinvar_clean_{snapshot}.csv.gz"
    cleaned_rows = sum(
        len(chunk)
        for chunk in pd.read_csv(
            cleaned_file,
            usecols=["VariationID"],
            dtype="string",
            chunksize=100_000
        )
    )

    assert checked_included == cleaned_rows, (
        f"{snapshot}: presence and cleaned counts disagree."
    )

    temporary.replace(presence_file)

    presence_summary.append({
        "snapshot": snapshot,
        "presence_rows": total_rows,
        "included_rows": status_counts[True],
        "excluded_rows": status_counts[False],
        "rows_without_id": missing_id_rows
    })

    print(f"  {snapshot}: saved and verified.")

display(pd.DataFrame(presence_summary))

Recording presence for 2022-01...
  2022-01: saved and verified.
Recording presence for 2023-01...
  2023-01: saved and verified.
Recording presence for 2024-01...
  2024-01: saved and verified.
Recording presence for 2025-01...
  2025-01: saved and verified.
Recording presence for 2026-01...
  2026-01: saved and verified.
Recording presence for 2026-09...
  2026-09: saved and verified.


,snapshot,presence_rows,included_rows,excluded_rows,rows_without_id
0,2022-01,1128266,1126984,1282,0
1,2023-01,1596737,1595467,1270,0
2,2024-01,2366650,2365518,1132,0
3,2025-01,3059494,3057876,1618,0
4,2026-01,4156969,3910144,246825,0
5,2026-09,4490877,4244291,246586,0


In [27]:
duplicate_summary = []
duplicate_details = []

for snapshot in snapshots:
    cleaned_file = processed_folder / f"clinvar_clean_{snapshot}.csv.gz"
    print(f"Checking repeated IDs in {snapshot}...")

    # First pass: count how often each variant ID appears.
    id_counts = Counter()

    for chunk in pd.read_csv(
        cleaned_file,
        usecols=["VariationID"],
        dtype="string",
        chunksize=100_000
    ):
        id_counts.update(chunk["VariationID"])

    repeated_ids = {
        variant_id
        for variant_id, count in id_counts.items()
        if count > 1
    }

    total_rows = sum(id_counts.values())
    unique_variants = len(id_counts)
    del id_counts

    if not repeated_ids:
        duplicate_summary.append({
            "snapshot": snapshot,
            "unique_variants": unique_variants,
            "repeated_ids": 0,
            "extra_rows": 0,
            "identical_extra_rows": 0,
            "ids_with_different_states": 0,
            "ids_with_different_labels": 0
        })
        continue

    # Second pass: collect only rows belonging to repeated IDs.
    repeated_parts = []

    for chunk in pd.read_csv(
        cleaned_file,
        dtype="string",
        chunksize=100_000
    ):
        matches = chunk.loc[chunk["VariationID"].isin(repeated_ids)]

        if not matches.empty:
            repeated_parts.append(matches.copy())

    repeated_rows = pd.concat(repeated_parts, ignore_index=True)
    groups = repeated_rows.groupby("VariationID", sort=False)

    state_counts = groups["clinical_state"].nunique(dropna=False)
    label_counts = groups["ClinicalSignificance"].nunique(dropna=False)

    duplicate_summary.append({
        "snapshot": snapshot,
        "unique_variants": unique_variants,
        "repeated_ids": len(repeated_ids),
        "extra_rows": total_rows - unique_variants,
        "identical_extra_rows": int(repeated_rows.duplicated().sum()),
        "ids_with_different_states": int((state_counts > 1).sum()),
        "ids_with_different_labels": int((label_counts > 1).sum())
    })

    # Record exactly which columns differ within each repeated ID.
    column_counts = groups.nunique(dropna=False)

    for variant_id, counts in column_counts.iterrows():
        differing_columns = counts.index[counts > 1].tolist()

        duplicate_details.append({
            "snapshot": snapshot,
            "VariationID": variant_id,
            "differing_columns": (
                ", ".join(differing_columns)
                if differing_columns
                else "none — identical rows"
            )
        })

    del repeated_parts, repeated_rows, groups, column_counts

duplicate_summary = pd.DataFrame(duplicate_summary)
duplicate_details = pd.DataFrame(
    duplicate_details,
    columns=["snapshot", "VariationID", "differing_columns"]
)

print("\nDUPLICATE SUMMARY")
print(duplicate_summary.to_string(index=False))

print("\nMOST COMMON DIFFERENCES")
if duplicate_details.empty:
    print("No repeated IDs found.")
else:
    difference_summary = (
        duplicate_details
        .groupby(["snapshot", "differing_columns"])
        .size()
        .reset_index(name="variant_count")
        .sort_values(
            ["snapshot", "variant_count"],
            ascending=[True, False]
        )
    )
    print(difference_summary.to_string(index=False))

Checking repeated IDs in 2022-01...
Checking repeated IDs in 2023-01...
Checking repeated IDs in 2024-01...
Checking repeated IDs in 2025-01...
Checking repeated IDs in 2026-01...
Checking repeated IDs in 2026-09...

DUPLICATE SUMMARY
snapshot  unique_variants  repeated_ids  extra_rows  identical_extra_rows  ids_with_different_states  ids_with_different_labels
 2022-01          1126575           409         409                     0                          0                          0
 2023-01          1594922           545         545                     0                          0                          0
 2024-01          2364896           622         622                     0                          0                          0
 2025-01          3057166           710         710                     0                          0                          0
 2026-01          3909372           772         772                     0                          0                         

In [28]:
import gzip

# Keep location details in the original cleaned files.
variant_columns = [
    column for column in CORE_COLUMNS
    if column not in ["Chromosome", "Start", "Stop"]
] + ["snapshot", "clinical_state"]

# Confirm that the earlier audit found only location differences.
allowed_differences = {"Chromosome", "Start", "Stop"}

for differences in duplicate_details["differing_columns"]:
    if differences != "none — identical rows":
        assert set(differences.split(", ")).issubset(allowed_differences), (
            f"Unexpected differences: {differences}"
        )

variant_summary = []

for snapshot in snapshots:
    source_file = processed_folder / f"clinvar_clean_{snapshot}.csv.gz"
    destination = processed_folder / f"clinvar_variants_{snapshot}.csv.gz"
    temporary = processed_folder / f"clinvar_variants_{snapshot}.partial.csv.gz"

    if destination.exists():
        print(f"{snapshot}: variant file already exists, skipped.")
        continue

    print(f"Creating variant records for {snapshot}...")

    seen_ids = set()
    rows_read = 0
    rows_saved = 0

    with gzip.open(temporary, "wt", encoding="utf-8", newline="") as handle:
        pd.DataFrame(columns=variant_columns).to_csv(handle, index=False)

        for chunk in pd.read_csv(
            source_file,
            usecols=variant_columns,
            dtype="string",
            chunksize=100_000
        ):
            rows_read += len(chunk)
            chunk = chunk[variant_columns]

            # Handle repeated IDs within this chunk and earlier chunks.
            unique_rows = chunk.drop_duplicates(subset=["VariationID"])
            unique_rows = unique_rows.loc[
                ~unique_rows["VariationID"].isin(seen_ids)
            ]

            unique_rows.to_csv(handle, index=False, header=False)

            seen_ids.update(unique_rows["VariationID"])
            rows_saved += len(unique_rows)

    expected_rows = int(
        duplicate_summary.loc[
            duplicate_summary["snapshot"].eq(snapshot),
            "unique_variants"
        ].iloc[0]
    )

    assert rows_saved == expected_rows, "Unexpected number of variants."
    del seen_ids

    # Read back the file and check uniqueness across chunk boundaries.
    checked_ids = set()
    checked_rows = 0

    for chunk in pd.read_csv(
        temporary,
        dtype="string",
        chunksize=100_000
    ):
        assert list(chunk.columns) == variant_columns
        assert chunk["snapshot"].eq(snapshot).all()
        assert chunk["VariationID"].notna().all()
        assert not chunk["VariationID"].duplicated().any()
        assert not chunk["VariationID"].isin(checked_ids).any()

        checked_ids.update(chunk["VariationID"])
        checked_rows += len(chunk)

    assert checked_rows == expected_rows
    del checked_ids

    temporary.replace(destination)

    variant_summary.append({
        "snapshot": snapshot,
        "cleaned_rows": rows_read,
        "unique_variants": rows_saved,
        "repeated_rows_collapsed": rows_read - rows_saved
    })

    print(f"  Saved and verified {rows_saved:,} unique variants.")

display(pd.DataFrame(variant_summary))

Creating variant records for 2022-01...
  Saved and verified 1,126,575 unique variants.
Creating variant records for 2023-01...
  Saved and verified 1,594,922 unique variants.
Creating variant records for 2024-01...
  Saved and verified 2,364,896 unique variants.
Creating variant records for 2025-01...
  Saved and verified 3,057,166 unique variants.
Creating variant records for 2026-01...
  Saved and verified 3,909,372 unique variants.
Creating variant records for 2026-09...
  Saved and verified 4,243,470 unique variants.


,snapshot,cleaned_rows,unique_variants,repeated_rows_collapsed
0,2022-01,1126984,1126575,409
1,2023-01,1595467,1594922,545
2,2024-01,2365518,2364896,622
3,2025-01,3057876,3057166,710
4,2026-01,3910144,3909372,772
5,2026-09,4244291,4243470,821


In [29]:
state_codes = {
    "absent_grch38": 0,
    "present_excluded": 1,
    "benign": 2,
    "pathogenic": 3,
    "vus": 4,
    "conflicting": 5,
    "other": 6
}

snapshot_states = []
timeline_counts = []

for snapshot in snapshots:
    print(f"Building timeline column for {snapshot}...")

    presence_file = processed_folder / f"clinvar_presence_{snapshot}.csv.gz"
    variant_file = processed_folder / f"clinvar_variants_{snapshot}.csv.gz"

    # Start with every variant present in this GRCh38 snapshot.
    presence_ids = pd.read_csv(
        presence_file,
        usecols=["VariationID"],
        dtype={"VariationID": "int64"}
    )["VariationID"].unique()

    states = pd.Series(
        state_codes["present_excluded"],
        index=pd.Index(presence_ids, name="VariationID"),
        name=snapshot,
        dtype="uint8"
    )

    del presence_ids
    included_variants = 0

    # Replace the default with the state of each retained variant.
    for chunk in pd.read_csv(
        variant_file,
        usecols=["VariationID", "clinical_state"],
        dtype={"VariationID": "int64", "clinical_state": "string"},
        chunksize=100_000
    ):
        assert chunk["VariationID"].isin(states.index).all(), (
            "A retained variant is missing from the presence file."
        )

        codes = chunk["clinical_state"].map(state_codes)
        assert codes.notna().all(), "Unexpected clinical state."
        assert codes.ge(2).all()

        states.loc[chunk["VariationID"]] = codes.to_numpy(dtype="uint8")
        included_variants += len(chunk)

    assert int(states.ge(2).sum()) == included_variants

    timeline_counts.append({
        "snapshot": snapshot,
        "present_variants": len(states),
        "retained_variants": included_variants,
        "excluded_variants": int(states.eq(1).sum())
    })

    snapshot_states.append(states)

# Align all snapshots by variant ID.
timeline = (
    pd.concat(snapshot_states, axis=1)
    .fillna(state_codes["absent_grch38"])
    .astype("uint8")
)

del snapshot_states, states

assert timeline.index.is_unique
assert list(timeline.columns) == snapshots
assert timeline.ne(0).any(axis=1).all()

# Turn codes into readable labels while keeping storage in memory compact.
state_names = list(state_codes)

for snapshot in snapshots:
    timeline[snapshot] = pd.Categorical.from_codes(
        timeline[snapshot].to_numpy(),
        categories=state_names
    )

print(f"\nTimelines built for {len(timeline):,} unique variants.")
display(pd.DataFrame(timeline_counts))
display(timeline.head(10))

Building timeline column for 2022-01...
Building timeline column for 2023-01...
Building timeline column for 2024-01...
Building timeline column for 2025-01...
Building timeline column for 2026-01...
Building timeline column for 2026-09...

Timelines built for 4,503,858 unique variants.


,snapshot,present_variants,retained_variants,excluded_variants
0,2022-01,1127855,1126575,1280
1,2023-01,1596189,1594922,1267
2,2024-01,2366025,2364896,1129
3,2025-01,3058778,3057166,1612
4,2026-01,4154664,3909372,245292
5,2026-09,4488519,4243470,245049


,2022-01,2023-01,2024-01,2025-01,2026-01,2026-09
VariationID,,,,,,
2,pathogenic,pathogenic,pathogenic,pathogenic,pathogenic,pathogenic
3,pathogenic,pathogenic,pathogenic,pathogenic,pathogenic,pathogenic
4,vus,vus,vus,vus,vus,vus
5,pathogenic,pathogenic,pathogenic,pathogenic,pathogenic,pathogenic
6,pathogenic,pathogenic,pathogenic,pathogenic,pathogenic,pathogenic
214885,vus,vus,conflicting,conflicting,conflicting,conflicting
9,conflicting,conflicting,conflicting,other,conflicting,other
10,conflicting,conflicting,conflicting,other,conflicting,conflicting
11,conflicting,conflicting,conflicting,conflicting,conflicting,conflicting


In [33]:
timeline_file = processed_folder / "clinvar_state_timelines.csv.gz"

checked_rows = 0
saved_counts = {snapshot: Counter() for snapshot in snapshots}

for chunk in pd.read_csv(
    timeline_file,
    dtype={"VariationID": "int64"},
    chunksize=100_000
):
    assert list(chunk.columns) == ["VariationID"] + snapshots

    # Compare each saved section with the same section in memory.
    expected = timeline.iloc[
        checked_rows : checked_rows + len(chunk)
    ]

    assert len(expected) == len(chunk), "The saved file has extra rows."

    assert (
        chunk["VariationID"].to_numpy() == expected.index.to_numpy()
    ).all(), "Saved variant IDs or their order differ."

    for snapshot in snapshots:
        assert chunk[snapshot].isin(state_names).all()

        assert (
            chunk[snapshot].to_numpy()
            == expected[snapshot].to_numpy()
        ).all(), f"Saved states differ in {snapshot}."

        saved_counts[snapshot].update(chunk[snapshot])

    checked_rows += len(chunk)

assert checked_rows == len(timeline), "The saved file has missing rows."

print("EXISTING TIMELINE VALIDATION PASSED")
print(f"Unique variants: {checked_rows:,}")
print(f"Snapshots: {len(snapshots)}")

display(
    pd.DataFrame(saved_counts)
    .reindex(state_names)
    .fillna(0)
    .astype("int64")
)

EXISTING TIMELINE VALIDATION PASSED
Unique variants: 4,503,858
Snapshots: 6


,2022-01,2023-01,2024-01,2025-01,2026-01,2026-09
absent_grch38,3376003,2907669,2137833,1445080,349194,15339
present_excluded,1280,1267,1129,1612,245292,245049
benign,477526,660712,862001,1147073,1283547,1373526
pathogenic,158261,201688,240652,288764,328540,354967
vus,421102,641852,1138685,1475196,2128771,2335946
conflicting,53940,73905,107267,129356,155362,166024
other,15746,16765,16291,16777,13152,13007


In [34]:
clinical_states = ["benign", "pathogenic", "vus", "conflicting"]

transition_summary = []
transition_tables = {}

for earlier, later in zip(snapshots[:-1], snapshots[1:]):
    before = timeline[earlier]
    after = timeline[later]

    # Include every state, including absence and excluded records.
    counts = pd.crosstab(before, after, dropna=False)
    counts = counts.reindex(
        index=state_names,
        columns=state_names,
        fill_value=0
    )

    assert int(counts.to_numpy().sum()) == len(timeline)
    transition_tables[f"{earlier}_to_{later}"] = counts

    comparable = (
        before.isin(clinical_states)
        & after.isin(clinical_states)
    )

    same_state = before.eq(after)
    changed = comparable & ~same_state
    unchanged = comparable & same_state

    absent_before = before.eq("absent_grch38")
    absent_after = after.eq("absent_grch38")

    appeared = absent_before & ~absent_after
    disappeared = ~absent_before & absent_after
    absent_both = absent_before & absent_after

    # Both observations exist, but at least one is excluded or "other".
    not_comparable = (
        ~absent_before
        & ~absent_after
        & ~comparable
    )

    comparable_count = int(comparable.sum())
    changed_count = int(changed.sum())

    # Every variant must belong to exactly one summary group.
    assert (
        int(unchanged.sum())
        + changed_count
        + int(appeared.sum())
        + int(disappeared.sum())
        + int(absent_both.sum())
        + int(not_comparable.sum())
    ) == len(timeline)

    transition_summary.append({
        "from": earlier,
        "to": later,
        "interval_months": (
            (int(later[:4]) - int(earlier[:4])) * 12
            + int(later[5:]) - int(earlier[5:])
        ),
        "comparable_variants": comparable_count,
        "unchanged_state": int(unchanged.sum()),
        "changed_state": changed_count,
        "changed_percent": (
            round(100 * changed_count / comparable_count, 2)
            if comparable_count else None
        ),
        "appeared_in_grch38": int(appeared.sum()),
        "absent_in_later_grch38": int(disappeared.sum()),
        "absent_in_both": int(absent_both.sum()),
        "present_but_not_comparable": int(not_comparable.sum()),
        "vus_to_benign": int(counts.loc["vus", "benign"]),
        "vus_to_pathogenic": int(counts.loc["vus", "pathogenic"]),
        "benign_to_pathogenic": int(counts.loc["benign", "pathogenic"]),
        "pathogenic_to_benign": int(counts.loc["pathogenic", "benign"])
    })

transition_summary = pd.DataFrame(transition_summary)

print("TRANSITION CHECKS PASSED")
print(transition_summary.to_string(index=False))

print("\nClinical-state transitions in the latest interval:")
latest_interval = f"{snapshots[-2]}_to_{snapshots[-1]}"

display(
    transition_tables[latest_interval].loc[
        clinical_states, clinical_states
    ]
)

TRANSITION CHECKS PASSED
   from      to  interval_months  comparable_variants  unchanged_state  changed_state  changed_percent  appeared_in_grch38  absent_in_later_grch38  absent_in_both  present_but_not_comparable  vus_to_benign  vus_to_pathogenic  benign_to_pathogenic  pathogenic_to_benign
2022-01 2023-01               12              1107951          1082903          25048             2.26              469853                    1519         2906150                       18385           1338               1135                    13                    73
2023-01 2024-01               12              1574266          1524056          50210             3.19              773618                    3782         2134051                       18141           9906               2049                    10                    43
2024-01 2025-01               12              2347768          2309588          38180             1.63              693353                     600         1444480      

2026-09,benign,pathogenic,vus,conflicting
2026-01,,,,
benign,1278259,9,214,3057
pathogenic,11,326488,312,1474
vus,1872,878,2109729,9627
conflicting,2491,942,407,151498


In [35]:
from datetime import datetime

# Confirm that the transition analysis completed.
assert len(transition_summary) == len(snapshots) - 1
assert len(transition_tables) == len(snapshots) - 1

# Use a new folder so previous reports are preserved.
run_time = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
report_folder = PROJECT_ROOT / "reports" / f"timeline_analysis_{run_time}"
report_folder.mkdir(parents=True, exist_ok=False)

transition_summary.to_csv(
    report_folder / "transition_summary.csv",
    index=False
)

# Save every state-to-state count in one table.
transition_parts = []

for interval, table in transition_tables.items():
    earlier, later = interval.split("_to_")

    records = (
        table.rename_axis(index="from_state", columns="to_state")
        .reset_index()
        .melt(
            id_vars="from_state",
            var_name="to_state",
            value_name="variant_count"
        )
    )

    records.insert(0, "from_snapshot", earlier)
    records.insert(1, "to_snapshot", later)
    transition_parts.append(records)

all_transitions = pd.concat(transition_parts, ignore_index=True)

all_transitions.to_csv(
    report_folder / "state_transition_counts.csv",
    index=False
)

# Check that each saved interval accounts for every variant.
saved_transitions = pd.read_csv(
    report_folder / "state_transition_counts.csv"
)

interval_totals = saved_transitions.groupby(
    ["from_snapshot", "to_snapshot"]
)["variant_count"].sum()

assert len(interval_totals) == len(snapshots) - 1
assert interval_totals.eq(len(timeline)).all()

checkpoint = f"""GeneVISTA preparation checkpoint

Unique variants in timeline: {len(timeline):,}
Snapshots: {", ".join(snapshots)}

Completed:
- Cleaned and verified six ClinVar snapshots.
- Preserved GRCh38 presence, including excluded records.
- Inspected repeated IDs and retained original location records.
- Created one retained record per variant per snapshot.
- Built and verified the state timeline.
- Calculated transitions between consecutive snapshots.

Interpretation:
- absent_grch38 means absent from this snapshot's GRCh38 records.
- present_excluded means present but excluded by the cleaning rules.
- Clinical labels are grouped; some label changes are not captured.
- The final interval is eight months; earlier intervals are twelve.
- Full-history transition reports are descriptive, not prediction features.

Remaining:
- Prepare HPO and ClinGen data for integration.
- Define prediction targets and chronological evaluation splits.
- Build features using only information available at each cutoff.
- Construct the graph, train models, and evaluate results.
"""

(report_folder / "checkpoint.txt").write_text(
    checkpoint,
    encoding="utf-8"
)

print("CHECKPOINT SAVED")
print(f"Reports folder: {report_folder}")

# Locate the next datasets without loading them.
print("\nHPO and ClinGen files available:")

raw_folder = PROJECT_ROOT / "data" / "raw"

for file in sorted(raw_folder.rglob("*")):
    if file.is_file():
        relative_path = file.relative_to(raw_folder)

        if any(
            keyword in str(relative_path).lower()
            for keyword in ["hpo", "clingen"]
        ):
            print(relative_path)

CHECKPOINT SAVED
Reports folder: D:\GeneVISTA\reports\timeline_analysis_20260913_001124_053037

HPO and ClinGen files available:
clingen\gene_disease_validity.csv
hpo\genes_to_phenotype.txt
hpo\hp.obo
hpo\phenotype.hpoa


In [36]:
from pathlib import Path
import gzip

PROJECT_ROOT = Path(r"D:\GeneVISTA")
raw_folder = PROJECT_ROOT / "data" / "raw"

if not raw_folder.exists():
    raise FileNotFoundError(f"Raw data folder not found: {raw_folder}")

supporting_files = [
    file
    for file in sorted(raw_folder.rglob("*"))
    if file.is_file()
    and any(
        name in str(file.relative_to(raw_folder)).lower()
        for name in ["hpo", "clingen"]
    )
]

if not supporting_files:
    print("No files matched HPO or ClinGen.")
    print("\nFolders directly inside data/raw:")

    for folder in sorted(raw_folder.iterdir()):
        if folder.is_dir():
            print(folder.name)

for file in supporting_files:
    print("\n" + "=" * 70)
    print(f"File: {file.relative_to(raw_folder)}")
    print(f"Size: {file.stat().st_size / (1024 ** 2):.2f} MB")

    text_extensions = {".csv", ".tsv", ".txt", ".obo", ".hpoa", ".json"}
    underlying_suffix = (
        Path(file.stem).suffix.lower()
        if file.suffix.lower() == ".gz"
        else file.suffix.lower()
    )

    if underlying_suffix not in text_extensions:
        print("Preview skipped: this file needs a different reader.")
        continue

    open_file = gzip.open if file.suffix.lower() == ".gz" else open

    print("First 12 lines:")

    with open_file(
        file,
        mode="rt",
        encoding="utf-8-sig",
        errors="replace"
    ) as handle:
        for line_number, line in enumerate(handle, start=1):
            print(line.rstrip()[:600])

            if line_number == 12:
                break


File: clingen\gene_disease_validity.csv
Size: 1.07 MB
First 12 lines:
"CLINGEN GENE DISEASE VALIDITY CURATIONS","","","","","","","","",""
"FILE CREATED: 2026-09-08","","","","","","","","",""
"WEBPAGE: https://search.clinicalgenome.org/kb/gene-validity","","","","","","","","",""
"+++++++++++","++++++++++++++","+++++++++++++","++++++++++++++++++","+++++++++","+++++++++","++++++++++++++","+++++++++++++","+++++++++++++++++++","+++++++++++++++++++"
"GENE SYMBOL","GENE ID (HGNC)","DISEASE LABEL","DISEASE ID (MONDO)","MOI","SOP","CLASSIFICATION","ONLINE REPORT","CLASSIFICATION DATE","GCEP"
"+++++++++++","++++++++++++++","+++++++++++++","++++++++++++++++++","+++++++++","+++++++++","++++++++++++++","+++++++++++++","+++++++++++++++++++","+++++++++++++++++++"
"AARS1","HGNC:20","Charcot-Marie-Tooth disease axonal type 2N","MONDO:0013212","AD","SOP10","Definitive","https://search.clinicalgenome.org/kb/gene-validity/CGGV:assertion_92de3832-c272-4993-8586-288c6331dec2-2024-03-14T160000.000Z","202

In [39]:
import csv
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"D:\GeneVISTA")
processed_folder = PROJECT_ROOT / "data" / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)

source_file = PROJECT_ROOT / "data" / "raw" / "clingen" / "gene_disease_validity.csv"

# Find the real header instead of assuming its line number.
header_line = None

with source_file.open(encoding="utf-8-sig", newline="") as handle:
    for line_number, row in enumerate(csv.reader(handle)):
        if row and row[0].strip() == "GENE SYMBOL":
            header_line = line_number
            break

if header_line is None:
    raise ValueError("Could not find the ClinGen column header.")

clingen = pd.read_csv(
    source_file,
    skiprows=header_line,
    dtype="string",
    encoding="utf-8-sig"
)

clingen.columns = clingen.columns.str.strip()

column_names = {
    "GENE SYMBOL": "gene_symbol",
    "GENE ID (HGNC)": "hgnc_id",
    "DISEASE LABEL": "disease_name",
    "DISEASE ID (MONDO)": "mondo_id",
    "MOI": "mode_of_inheritance",
    "SOP": "sop",
    "CLASSIFICATION": "classification",
    "ONLINE REPORT": "report_url",
    "CLASSIFICATION DATE": "classification_date",
    "GCEP": "expert_panel"
}

assert set(column_names).issubset(clingen.columns), (
    "Expected ClinGen columns are missing."
)

clingen = clingen.rename(columns=column_names)

for column in clingen.columns:
    clingen[column] = clingen[column].str.strip()

# Remove the decorative separator, not evidence records.
separator = clingen["gene_symbol"].str.fullmatch(r"\++", na=False)
clingen = clingen.loc[~separator].copy()
original_rows = len(clingen)

clingen = clingen.replace({"": pd.NA, "-": pd.NA})

# Remove only rows that are identical across every column.
exact_duplicates = int(clingen.duplicated().sum())
clingen = clingen.drop_duplicates().reset_index(drop=True)

# Stop for inspection if an identifier is missing or malformed.
assert clingen["gene_symbol"].notna().all(), "Missing gene symbols."
assert clingen["hgnc_id"].str.fullmatch(
    r"HGNC:\d+", na=False
).all(), "Missing or unexpected HGNC IDs."
assert clingen["mondo_id"].str.fullmatch(
    r"MONDO:\d+", na=False
).all(), "Missing or unexpected MONDO IDs."
assert clingen["classification"].notna().all(), "Missing classifications."



# Preserve the source date text; add a normalized date for filtering.
clingen["classification_date_iso"] = (
    dates.dt.strftime("%Y-%m-%d").astype("string")
)
clingen["source_file_created"] = "2026-09-08"

print("CLINGEN CLEANING CHECKS PASSED")
print(f"Original evidence rows: {original_rows:,}")
print(f"Exact duplicate rows removed: {exact_duplicates:,}")
print(f"Retained evidence rows: {len(clingen):,}")
print(f"Unique genes: {clingen['hgnc_id'].nunique():,}")
print(f"Unique diseases: {clingen['mondo_id'].nunique():,}")
print(f"Missing classification dates: {dates.isna().sum():,}")

print("\nClassification counts:")
print(clingen["classification"].value_counts().to_string())

display(clingen.head())

CLINGEN CLEANING CHECKS PASSED
Original evidence rows: 3,670
Exact duplicate rows removed: 0
Retained evidence rows: 3,670
Unique genes: 3,029
Unique diseases: 2,145
Missing classification dates: 0

Classification counts:
classification
Definitive                       2292
Limited                           546
Moderate                          454
Disputed                          197
Strong                             84
Refuted                            49
No Known Disease Relationship      48


,gene_symbol,hgnc_id,disease_name,mondo_id,mode_of_inheritance,sop,classification,report_url,classification_date,expert_panel,classification_date_iso,source_file_created
0,AARS1,HGNC:20,Charcot-Marie-Tooth disease axonal type 2N,MONDO:0013212,AD,SOP10,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2024-03-14T16:00:00.000Z,Charcot-Marie-Tooth Disease Gene Curation Expe...,2024-03-14,2026-09-08
1,AARS2,HGNC:21022,mitochondrial disease,MONDO:0044970,AR,SOP8,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2022-04-18T16:00:00.000Z,Mitochondrial Diseases Gene Curation Expert Panel,2022-04-18,2026-09-08
2,ABCA3,HGNC:33,interstitial lung disease due to ABCA3 deficiency,MONDO:0012582,AR,SOP10,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2024-09-17T16:00:00.000Z,Interstitial Lung Disease Gene Curation Expert...,2024-09-17,2026-09-08
3,ABCA4,HGNC:34,ABCA4-related retinopathy,MONDO:0800406,AR,SOP9,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2022-10-06T16:00:00.000Z,Retina Gene Curation Expert Panel,2022-10-06,2026-09-08
4,ABCB4,HGNC:45,progressive familial intrahepatic cholestasis ...,MONDO:0011214,AR,SOP9,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2022-11-23T17:00:00.000Z,General Gene Curation Expert Panel,2022-11-23,2026-09-08


In [40]:
# Parse each value separately so different date formats can coexist.
dates = clingen["classification_date"].map(
    lambda value: pd.to_datetime(value, errors="coerce", utc=True)
    if pd.notna(value)
    else pd.NaT
)

dates = pd.to_datetime(dates, errors="coerce", utc=True)

invalid_dates = (
    clingen["classification_date"].notna()
    & dates.isna()
)

if invalid_dates.any():
    print("Date values that still need inspection:")
    print(
        clingen.loc[invalid_dates, "classification_date"]
        .value_counts()
        .to_string()
    )

    raise ValueError("Unrecognized dates remain. Share the values printed above.")

print("All non-missing classification dates parsed successfully.")

All non-missing classification dates parsed successfully.


In [42]:
source_file = (
    PROJECT_ROOT / "data" / "raw" / "hpo" / "genes_to_phenotype.txt"
)

gene_phenotypes = pd.read_csv(
    source_file,
    sep="\t",
    dtype="string",
    keep_default_na=False
)

expected_columns = [
    "ncbi_gene_id",
    "gene_symbol",
    "hpo_id",
    "hpo_name",
    "frequency",
    "disease_id"
]

assert set(expected_columns).issubset(gene_phenotypes.columns), (
    "Expected HPO columns are missing."
)

for column in gene_phenotypes.columns:
    gene_phenotypes[column] = gene_phenotypes[column].str.strip()

gene_phenotypes = gene_phenotypes.replace({
    "": pd.NA,
    "-": pd.NA
})

original_rows = len(gene_phenotypes)
duplicate_rows = int(gene_phenotypes.duplicated().sum())

gene_phenotypes = (
    gene_phenotypes
    .drop_duplicates()
    .reset_index(drop=True)
)

# Keep identifiers as strings so their original format is preserved.
assert gene_phenotypes["ncbi_gene_id"].str.fullmatch(
    r"[0-9]+", na=False
).all(), "Missing or unexpected gene IDs."

assert gene_phenotypes["hpo_id"].str.fullmatch(
    r"HP:[0-9]{7}", na=False
).all(), "Missing or unexpected HPO IDs."

# Keep rows with missing symbols; their gene IDs are valid.
missing_symbols = gene_phenotypes["gene_symbol"].isna()

print(f"Rows missing gene symbols: {missing_symbols.sum():,}")
print(
    "Affected gene IDs:",
    gene_phenotypes.loc[missing_symbols, "ncbi_gene_id"].nunique()
)

display(
    gene_phenotypes.loc[missing_symbols, ["ncbi_gene_id", "hpo_id"]]
    .head(10)
)

for column in ["hpo_name", "disease_id"]:
    assert gene_phenotypes[column].notna().all(), (
        f"Missing values in {column}."
    )

print("GENE–PHENOTYPE CLEANING CHECKS PASSED")
print(f"Original rows: {original_rows:,}")
print(f"Exact duplicates removed: {duplicate_rows:,}")
print(f"Retained rows: {len(gene_phenotypes):,}")
print(f"Unique genes: {gene_phenotypes['ncbi_gene_id'].nunique():,}")
print(f"Unique HPO terms: {gene_phenotypes['hpo_id'].nunique():,}")
print(f"Unique diseases: {gene_phenotypes['disease_id'].nunique():,}")

print("\nDisease ID prefixes:")
print(
    gene_phenotypes["disease_id"]
    .str.split(":").str[0]
    .value_counts()
    .to_string()
)

display(gene_phenotypes.head())

Rows missing gene symbols: 375
Affected gene IDs: 7


,ncbi_gene_id,hpo_id
77568,3653,HP:0001159
77569,3653,HP:0007328
77570,3653,HP:0003745
77571,3653,HP:0001290
77572,3653,HP:0001270
77573,3653,HP:0001250
77574,3653,HP:0001249
77575,3653,HP:0002591
77576,3653,HP:0001263
77577,3653,HP:0001262


GENE–PHENOTYPE CLEANING CHECKS PASSED
Original rows: 333,983
Exact duplicates removed: 0
Retained rows: 333,983
Unique genes: 5,283
Unique HPO terms: 10,563
Unique diseases: 9,177

Disease ID prefixes:
disease_id
ORPHA    171700
OMIM     162283


,ncbi_gene_id,gene_symbol,hpo_id,hpo_name,frequency,disease_id
0,10,NAT2,HP:0000007,Autosomal recessive inheritance,<NA>,OMIM:243400
1,10,NAT2,HP:0001939,Abnormality of metabolism/homeostasis,<NA>,OMIM:243400
2,16,AARS1,HP:0002460,Distal muscle weakness,15/15,OMIM:613287
3,16,AARS1,HP:0002451,Limb dystonia,3/3,OMIM:616339
4,16,AARS1,HP:0008619,Bilateral sensorineural hearing impairment,HP:0040283,ORPHA:33364


In [43]:
output_file = processed_folder / "hpo_gene_phenotype_clean.csv.gz"
temporary = processed_folder / "hpo_gene_phenotype_clean.partial.csv.gz"

if output_file.exists():
    file_to_check = output_file
else:
    gene_phenotypes.to_csv(temporary, index=False, compression="gzip")
    file_to_check = temporary

saved_data = pd.read_csv(
    file_to_check,
    dtype="string",
    keep_default_na=False
).replace({"": pd.NA})

pd.testing.assert_frame_equal(
    saved_data,
    gene_phenotypes,
    check_dtype=False
)

if file_to_check != output_file:
    temporary.replace(output_file)

print("GENE–PHENOTYPE FILE VERIFIED")
print(f"Saved to: {output_file}")

GENE–PHENOTYPE FILE VERIFIED
Saved to: D:\GeneVISTA\data\processed\hpo_gene_phenotype_clean.csv.gz


In [44]:
import json

ontology_file = PROJECT_ROOT / "data" / "raw" / "hpo" / "hp.obo"

terms = []
relationships = []
current_term = None
ontology_version = None


def save_term(term):
    if term is None or "id" not in term:
        return

    terms.append({
        "hpo_id": term["id"],
        "hpo_name": term.get("name", ""),
        "is_obsolete": term.get("is_obsolete", "false"),
        "alternative_ids": json.dumps(term.get("alt_id", [])),
        "replaced_by": json.dumps(term.get("replaced_by", [])),
        "consider": json.dumps(term.get("consider", []))
    })

    for parent_id in term.get("is_a", []):
        relationships.append({
            "child_hpo_id": term["id"],
            "parent_hpo_id": parent_id,
            "relationship": "is_a"
        })


with ontology_file.open(encoding="utf-8") as handle:
    for raw_line in handle:
        line = raw_line.strip()

        if line.startswith("data-version:"):
            ontology_version = line.split(": ", 1)[1]

        if line.startswith("[") and line.endswith("]"):
            save_term(current_term)
            current_term = {} if line == "[Term]" else None
            continue

        if current_term is None or ": " not in line:
            continue

        key, value = line.split(": ", 1)

        if key in ["id", "name", "is_obsolete"]:
            current_term[key] = value

        elif key in ["alt_id", "is_a", "replaced_by", "consider"]:
            # Keep the ID, excluding any trailing comment.
            identifier = value.split()[0]
            current_term.setdefault(key, []).append(identifier)

save_term(current_term)

hpo_terms = pd.DataFrame(terms, dtype="string")
hpo_edges = pd.DataFrame(
    relationships,
    columns=["child_hpo_id", "parent_hpo_id", "relationship"],
    dtype="string"
).drop_duplicates().reset_index(drop=True)

assert ontology_version, "Ontology version not found."
assert not hpo_terms.empty
assert hpo_terms["hpo_id"].is_unique
assert hpo_terms["hpo_id"].str.fullmatch(r"HP:[0-9]{7}", na=False).all()
assert hpo_terms["hpo_name"].ne("").all()
assert hpo_terms["is_obsolete"].isin(["true", "false"]).all()

term_ids = set(hpo_terms["hpo_id"])

assert hpo_edges["child_hpo_id"].isin(term_ids).all()
assert hpo_edges["parent_hpo_id"].isin(term_ids).all()
assert not hpo_edges["child_hpo_id"].eq(hpo_edges["parent_hpo_id"]).any()

hpo_terms["ontology_version"] = ontology_version
hpo_edges["ontology_version"] = ontology_version

# Check whether gene annotations use primary, alternative, or unknown IDs.
alternative_ids = {
    alternative_id
    for values in hpo_terms["alternative_ids"]
    for alternative_id in json.loads(values)
}

annotation_ids = set(gene_phenotypes["hpo_id"].dropna())
unknown_ids = annotation_ids - term_ids - alternative_ids
alternative_only = (annotation_ids - term_ids) & alternative_ids

print("HPO ONTOLOGY CHECKS PASSED")
print(f"Version: {ontology_version}")
print(f"Terms: {len(hpo_terms):,}")
print(f"Obsolete terms: {hpo_terms['is_obsolete'].eq('true').sum():,}")
print(f"Parent–child links: {len(hpo_edges):,}")
print(f"Annotation IDs using alternative IDs: {len(alternative_only):,}")
print(f"Annotation IDs not found in this ontology: {len(unknown_ids):,}")

if unknown_ids:
    print("Unknown ID examples:", sorted(unknown_ids)[:20])

display(hpo_terms.head())
display(hpo_edges.head())

HPO ONTOLOGY CHECKS PASSED
Version: hp/releases/2026-09-01
Terms: 20,482
Obsolete terms: 588
Parent–child links: 24,436
Annotation IDs using alternative IDs: 0
Annotation IDs not found in this ontology: 0


,hpo_id,hpo_name,is_obsolete,alternative_ids,replaced_by,consider,ontology_version
0,HP:0000001,All,false,[],[],[],hp/releases/2026-09-01
1,HP:0000002,Abnormality of body height,false,[],[],[],hp/releases/2026-09-01
2,HP:0000003,Multicystic kidney dysplasia,false,"[""HP:0004715""]",[],[],hp/releases/2026-09-01
3,HP:0000005,Mode of inheritance,false,"[""HP:0001425"", ""HP:0001453"", ""HP:0001461"", ""HP...",[],[],hp/releases/2026-09-01
4,HP:0000006,Autosomal dominant inheritance,false,"[""HP:0001415"", ""HP:0001447"", ""HP:0001448"", ""HP...",[],[],hp/releases/2026-09-01


,child_hpo_id,parent_hpo_id,relationship,ontology_version
0,HP:0000002,HP:0001507,is_a,hp/releases/2026-09-01
1,HP:0000003,HP:0000107,is_a,hp/releases/2026-09-01
2,HP:0000005,HP:0000001,is_a,hp/releases/2026-09-01
3,HP:0000006,HP:0034345,is_a,hp/releases/2026-09-01
4,HP:0000007,HP:0034345,is_a,hp/releases/2026-09-01


In [45]:
tables_to_save = {
    "hpo_terms.csv.gz": hpo_terms,
    "hpo_parent_child.csv.gz": hpo_edges
}

for filename, table in tables_to_save.items():
    destination = processed_folder / filename
    temporary = processed_folder / filename.replace(".csv.gz", ".partial.csv.gz")

    if destination.exists():
        file_to_check = destination
    else:
        table.to_csv(temporary, index=False, compression="gzip")
        file_to_check = temporary

    saved_table = pd.read_csv(
        file_to_check,
        dtype="string",
        keep_default_na=False
    )

    pd.testing.assert_frame_equal(
        saved_table,
        table.reset_index(drop=True),
        check_dtype=False
    )

    if file_to_check != destination:
        temporary.replace(destination)

    print(f"Verified: {filename}")

Verified: hpo_terms.csv.gz
Verified: hpo_parent_child.csv.gz


In [46]:
source_file = PROJECT_ROOT / "data" / "raw" / "hpo" / "phenotype.hpoa"

header_line = None
annotation_version = None

# Read the metadata and locate the actual table header.
with source_file.open(encoding="utf-8-sig") as handle:
    for line_number, line in enumerate(handle):
        if line.startswith("#version:"):
            annotation_version = line.split(":", 1)[1].strip()

        if line.startswith("database_id\t"):
            header_line = line_number
            break

if header_line is None:
    raise ValueError("Disease–phenotype header not found.")

disease_phenotypes = pd.read_csv(
    source_file,
    sep="\t",
    skiprows=header_line,
    dtype="string",
    keep_default_na=False
)

required = [
    "database_id", "disease_name", "qualifier", "hpo_id",
    "reference", "evidence", "onset", "frequency",
    "sex", "modifier", "aspect", "biocuration"
]

assert set(required).issubset(disease_phenotypes.columns)

for column in disease_phenotypes.columns:
    disease_phenotypes[column] = disease_phenotypes[column].str.strip()

original_rows = len(disease_phenotypes)
duplicate_rows = int(disease_phenotypes.duplicated().sum())

disease_phenotypes = (
    disease_phenotypes
    .drop_duplicates()
    .reset_index(drop=True)
)

assert disease_phenotypes["database_id"].ne("").all()
assert disease_phenotypes["hpo_id"].str.fullmatch(
    r"HP:[0-9]{7}", na=False
).all()

# Preserve the original qualifier and add an explicit interpretation.
# Unexpected qualifiers stay unresolved instead of becoming positive links.
qualifiers = disease_phenotypes["qualifier"].str.upper()

disease_phenotypes["annotation_polarity"] = (
    qualifiers.map({
        "": "positive",
        "NOT": "negative"
    })
    .fillna("unresolved")
    .astype("string")
)

primary_ids = set(hpo_terms["hpo_id"])
obsolete_ids = set(
    hpo_terms.loc[hpo_terms["is_obsolete"].eq("true"), "hpo_id"]
)

alternative_ids = {
    alternative_id
    for values in hpo_terms["alternative_ids"]
    for alternative_id in json.loads(values)
}

# Report term status without dropping or automatically replacing annotations.
disease_phenotypes["term_status"] = "unknown"

is_primary = disease_phenotypes["hpo_id"].isin(primary_ids)
is_alternative = disease_phenotypes["hpo_id"].isin(alternative_ids)
is_obsolete = disease_phenotypes["hpo_id"].isin(obsolete_ids)

disease_phenotypes.loc[
    is_alternative & ~is_primary, "term_status"
] = "alternative"

disease_phenotypes.loc[is_primary, "term_status"] = "current"
disease_phenotypes.loc[is_obsolete, "term_status"] = "obsolete"

disease_phenotypes["term_status"] = (
    disease_phenotypes["term_status"].astype("string")
)

assert annotation_version, "Annotation version not found."
disease_phenotypes["annotation_version"] = annotation_version

print("DISEASE–PHENOTYPE CHECKS PASSED")
print(f"Annotation version: {annotation_version}")
print(f"Original rows: {original_rows:,}")
print(f"Exact duplicates removed: {duplicate_rows:,}")
print(f"Retained rows: {len(disease_phenotypes):,}")
print(f"Unique diseases: {disease_phenotypes['database_id'].nunique():,}")

print("\nAnnotation polarity:")
print(disease_phenotypes["annotation_polarity"].value_counts().to_string())

print("\nHPO term status:")
print(disease_phenotypes["term_status"].value_counts().to_string())

print("\nAnnotation aspects:")
print(disease_phenotypes["aspect"].value_counts(dropna=False).to_string())

unresolved = disease_phenotypes["annotation_polarity"].eq("unresolved")

if unresolved.any():
    print("\nQualifiers needing inspection:")
    print(
        disease_phenotypes.loc[unresolved, "qualifier"]
        .value_counts()
        .to_string()
    )

display(disease_phenotypes.head())

DISEASE–PHENOTYPE CHECKS PASSED
Annotation version: 2026-09-02
Original rows: 286,651
Exact duplicates removed: 31
Retained rows: 286,620
Unique diseases: 12,880

Annotation polarity:
annotation_polarity
positive    285887
negative       733

HPO term status:
term_status
current    286620

Annotation aspects:
aspect
P    268875
I      8978
C      8559
H       131
M        77


,database_id,disease_name,qualifier,hpo_id,reference,evidence,onset,frequency,sex,modifier,aspect,biocuration,annotation_polarity,term_status,annotation_version
0,OMIM:619340,Developmental and epileptic encephalopathy 96,,HP:0011097,PMID:31675180,PCS,,1/2,,,P,HPO:probinson[2021-06-21],positive,current,2026-09-02
1,OMIM:619340,Developmental and epileptic encephalopathy 96,,HP:0002187,PMID:31675180,PCS,,1/1,,,P,HPO:probinson[2021-06-21],positive,current,2026-09-02
2,OMIM:619340,Developmental and epileptic encephalopathy 96,,HP:0001518,PMID:31675180,PCS,,1/2,,,P,HPO:probinson[2021-06-21],positive,current,2026-09-02
3,OMIM:619340,Developmental and epileptic encephalopathy 96,,HP:0032792,PMID:31675180,PCS,,1/2,,,P,HPO:probinson[2021-06-21],positive,current,2026-09-02
4,OMIM:619340,Developmental and epileptic encephalopathy 96,,HP:0011451,PMID:31675180,PCS,,1/2,,,P,HPO:probinson[2021-06-21],positive,current,2026-09-02


In [47]:
destination = processed_folder / "hpo_disease_phenotype_clean.csv.gz"
temporary = processed_folder / "hpo_disease_phenotype_clean.partial.csv.gz"

if destination.exists():
    file_to_check = destination
else:
    disease_phenotypes.to_csv(temporary, index=False, compression="gzip")
    file_to_check = temporary

saved_annotations = pd.read_csv(
    file_to_check,
    dtype="string",
    keep_default_na=False
)

pd.testing.assert_frame_equal(
    saved_annotations,
    disease_phenotypes,
    check_dtype=False
)

if file_to_check != destination:
    temporary.replace(destination)

print("DISEASE–PHENOTYPE FILE VERIFIED")
print(f"Saved to: {destination}")

DISEASE–PHENOTYPE FILE VERIFIED
Saved to: D:\GeneVISTA\data\processed\hpo_disease_phenotype_clean.csv.gz


In [48]:
# Show the coverage before selecting graph records.
print("ANNOTATION COVERAGE")
print(
    disease_phenotypes.groupby(
        ["annotation_polarity", "term_status", "aspect"],
        dropna=False
    ).size().to_string()
)

eligible = (
    disease_phenotypes["term_status"].eq("current")
    & disease_phenotypes["aspect"].eq("P")
    & disease_phenotypes["annotation_polarity"].isin(
        ["positive", "negative"]
    )
)

graph_annotations = disease_phenotypes.loc[eligible].copy()

assert not graph_annotations.empty, "No eligible phenotype annotations."

# Count evidence records for each disease–phenotype pair.
pair_counts = (
    graph_annotations.groupby(
        ["database_id", "hpo_id", "annotation_polarity"]
    )
    .size()
    .unstack("annotation_polarity", fill_value=0)
    .reindex(columns=["positive", "negative"], fill_value=0)
    .reset_index()
    .rename(columns={
        "database_id": "disease_id",
        "positive": "positive_annotation_count",
        "negative": "negative_annotation_count"
    })
)

pair_counts.columns.name = None

has_positive = pair_counts["positive_annotation_count"].gt(0)
has_negative = pair_counts["negative_annotation_count"].gt(0)

pair_counts["evidence_status"] = "positive_only"
pair_counts.loc[has_negative, "evidence_status"] = "negative_only"
pair_counts.loc[
    has_positive & has_negative, "evidence_status"
] = "mixed"

pair_counts["annotation_version"] = (
    graph_annotations["annotation_version"].iloc[0]
)

assert not pair_counts.duplicated(["disease_id", "hpo_id"]).any()

assert (
    pair_counts["positive_annotation_count"].sum()
    + pair_counts["negative_annotation_count"].sum()
) == len(graph_annotations)

# Only unopposed positive pairs become positive graph links.
disease_phenotype_links = pair_counts.loc[
    pair_counts["evidence_status"].eq("positive_only")
].copy()

disease_phenotype_links["relationship"] = "has_phenotype"

# Keep the audit table too, including negative and mixed pairs.
tables = {
    "hpo_disease_phenotype_pair_audit.csv.gz": pair_counts,
    "hpo_disease_phenotype_links.csv.gz": disease_phenotype_links
}

for filename, table in tables.items():
    destination = processed_folder / filename
    temporary = processed_folder / filename.replace(
        ".csv.gz", ".partial.csv.gz"
    )

    if destination.exists():
        file_to_check = destination
    else:
        table.to_csv(temporary, index=False, compression="gzip")
        file_to_check = temporary

    saved_table = pd.read_csv(
        file_to_check,
        dtype="string",
        keep_default_na=False
    )

    expected_table = table.reset_index(drop=True).astype("string")

    pd.testing.assert_frame_equal(saved_table, expected_table)

    if file_to_check != destination:
        temporary.replace(destination)

    print(f"\nVerified: {filename}")

print("\nGRAPH LINK SUMMARY")
print(f"Eligible annotation records: {len(graph_annotations):,}")
print(f"Records outside this selection: {(~eligible).sum():,}")
print(f"Unique disease–phenotype pairs: {len(pair_counts):,}")
print(pair_counts["evidence_status"].value_counts().to_string())
print(f"Positive graph links: {len(disease_phenotype_links):,}")

display(disease_phenotype_links.head())

ANNOTATION COVERAGE
annotation_polarity  term_status  aspect
negative             current      H              7
                                  P            726
positive             current      C           8559
                                  H            124
                                  I           8978
                                  M             77
                                  P         268149

Verified: hpo_disease_phenotype_pair_audit.csv.gz

Verified: hpo_disease_phenotype_links.csv.gz

GRAPH LINK SUMMARY
Eligible annotation records: 268,875
Records outside this selection: 17,745
Unique disease–phenotype pairs: 268,306
evidence_status
positive_only    267580
negative_only       726
Positive graph links: 267,580


,disease_id,hpo_id,positive_annotation_count,negative_annotation_count,evidence_status,annotation_version,relationship
0,DECIPHER:1,HP:0000252,1,0,positive_only,2026-09-02,has_phenotype
1,DECIPHER:1,HP:0001249,1,0,positive_only,2026-09-02,has_phenotype
2,DECIPHER:1,HP:0001250,1,0,positive_only,2026-09-02,has_phenotype
3,DECIPHER:1,HP:0001252,1,0,positive_only,2026-09-02,has_phenotype
4,DECIPHER:1,HP:0001518,1,0,positive_only,2026-09-02,has_phenotype


In [49]:
latest_snapshot = snapshots[-1]
variant_file = (
    processed_folder / f"clinvar_variants_{latest_snapshot}.csv.gz"
)

gene_columns = ["GeneID", "GeneSymbol", "HGNC_ID"]
gene_parts = []

print(f"Reading gene identifiers from {latest_snapshot}...")

for chunk in pd.read_csv(
    variant_file,
    usecols=gene_columns,
    dtype="string",
    keep_default_na=False,
    chunksize=100_000
):
    for column in gene_columns:
        chunk[column] = chunk[column].str.strip()

    # Keep unique combinations, not millions of repeated variant rows.
    gene_parts.append(chunk.drop_duplicates())

clinvar_gene_records = (
    pd.concat(gene_parts, ignore_index=True)
    .drop_duplicates()
    .reset_index(drop=True)
)

del gene_parts

# Identify records containing one recognizable ID of each type.
single_ncbi = (
    clinvar_gene_records["GeneID"]
    .str.fullmatch(r"[1-9][0-9]*", na=False)
)

single_hgnc = (
    clinvar_gene_records["HGNC_ID"]
    .str.fullmatch(r"HGNC:[1-9][0-9]*", na=False)
)

mapping_candidates = (
    clinvar_gene_records.loc[
        single_ncbi & single_hgnc,
        ["GeneID", "HGNC_ID"]
    ]
    .drop_duplicates()
    .rename(columns={
        "GeneID": "ncbi_gene_id",
        "HGNC_ID": "hgnc_id"
    })
)

# Check whether either identifier points to multiple partners.
ncbi_partner_counts = (
    mapping_candidates.groupby("ncbi_gene_id")["hgnc_id"].nunique()
)

hgnc_partner_counts = (
    mapping_candidates.groupby("hgnc_id")["ncbi_gene_id"].nunique()
)

ambiguous_ncbi = set(ncbi_partner_counts[ncbi_partner_counts > 1].index)
ambiguous_hgnc = set(hgnc_partner_counts[hgnc_partner_counts > 1].index)

ambiguous_mapping = (
    mapping_candidates["ncbi_gene_id"].isin(ambiguous_ncbi)
    | mapping_candidates["hgnc_id"].isin(ambiguous_hgnc)
)

gene_id_map = mapping_candidates.loc[~ambiguous_mapping].copy()

assert gene_id_map["ncbi_gene_id"].is_unique
assert gene_id_map["hgnc_id"].is_unique

hpo_gene_ids = set(gene_phenotypes["ncbi_gene_id"])
clingen_gene_ids = set(clingen["hgnc_id"])

print("\nGENE IDENTIFIER SUMMARY")
print(f"Unique ClinVar gene-field combinations: {len(clinvar_gene_records):,}")
print(f"Candidate ID pairs: {len(mapping_candidates):,}")
print(f"Ambiguous pairs held aside: {ambiguous_mapping.sum():,}")
print(f"Unambiguous ID pairs: {len(gene_id_map):,}")

print(
    "HPO genes covered:",
    f"{len(hpo_gene_ids & set(gene_id_map['ncbi_gene_id'])):,}",
    "of",
    f"{len(hpo_gene_ids):,}"
)

print(
    "ClinGen genes covered:",
    f"{len(clingen_gene_ids & set(gene_id_map['hgnc_id'])):,}",
    "of",
    f"{len(clingen_gene_ids):,}"
)

print("\nExamples needing further parsing or missing an ID:")
display(
    clinvar_gene_records.loc[~(single_ncbi & single_hgnc)].head(15)
)

if ambiguous_mapping.any():
    print("\nAmbiguous mapping examples:")
    display(mapping_candidates.loc[ambiguous_mapping].head(15))

Reading gene identifiers from 2026-09...

GENE IDENTIFIER SUMMARY
Unique ClinVar gene-field combinations: 27,537
Candidate ID pairs: 19,015
Ambiguous pairs held aside: 0
Unambiguous ID pairs: 19,015
HPO genes covered: 5,252 of 5,283
ClinGen genes covered: 3,029 of 3,029

Examples needing further parsing or missing an ID:


,GeneID,GeneSymbol,HGNC_ID
34,-1,ATXN8;ATXN8OS;LOC109461478,-
152,-1,FANCI;POLG,-
190,-1,DDX25;HYLS1;PUS3,-
289,-1,ERCC8;NDUFAF2,-
330,-1,ADA;PKIG,-
344,-1,SUMO4;TAB2,-
348,-1,GCDH;KLF1;LOC117125594,-
356,-1,LLGL2;TSEN54,-
427,-1,ASPN;CENPP,-
437,-1,APOC4-APOC2;APOC2,-


In [54]:
# Preserve the source of this current-snapshot mapping.
gene_id_map = gene_id_map.copy()
gene_id_map["mapping_source"] = "ClinVar"
gene_id_map["mapping_snapshot"] = latest_snapshot

# HPO already provides NCBI IDs. Add HGNC IDs where available.
hpo_gene_evidence = gene_phenotypes.merge(
    gene_id_map,
    on="ncbi_gene_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

hpo_gene_evidence["mapping_status"] = (
    hpo_gene_evidence["_merge"]
    .map({
        "both": "mapped",
        "left_only": "unmapped",
        "right_only": "unexpected"
    })
    .astype("string")
)

hpo_gene_evidence = hpo_gene_evidence.drop(columns="_merge")

# ClinGen provides HGNC IDs. Add the corresponding NCBI IDs.
clingen_gene_evidence = clingen.merge(
    gene_id_map,
    on="hgnc_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

clingen_gene_evidence["mapping_status"] = (
    clingen_gene_evidence["_merge"]
    .map({
        "both": "mapped",
        "left_only": "unmapped",
        "right_only": "unexpected"
    })
    .astype("string")
)

clingen_gene_evidence = clingen_gene_evidence.drop(columns="_merge")

# A mapping join must not add or remove evidence records.
assert len(hpo_gene_evidence) == len(gene_phenotypes)
assert len(clingen_gene_evidence) == len(clingen)
assert clingen_gene_evidence["mapping_status"].eq("mapped").all()

unmapped_hpo_genes = (
    hpo_gene_evidence.loc[
        hpo_gene_evidence["mapping_status"].eq("unmapped"),
        ["ncbi_gene_id", "gene_symbol"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("GENE EVIDENCE JOINS PASSED")
print(f"HPO evidence rows retained: {len(hpo_gene_evidence):,}")
print(f"ClinGen evidence rows retained: {len(clingen_gene_evidence):,}")
print(
    "HPO genes without an HGNC mapping:",
    unmapped_hpo_genes["ncbi_gene_id"].nunique()
)

display(unmapped_hpo_genes)

GENE EVIDENCE JOINS PASSED
HPO evidence rows retained: 333,983
ClinGen evidence rows retained: 3,670
HPO genes without an HGNC mapping: 31


,ncbi_gene_id,gene_symbol
0,2464,FRA16E
1,3500,IGHG1
2,3653,<NA>
3,7467,<NA>
4,8284,KDM5D
5,9081,PRY
6,9082,XKRY
7,9083,BPY2
8,9085,CDY1
9,9426,CDY2A


In [55]:
tables_to_save = {
    "gene_id_map_clinvar_2026-09.csv.gz": gene_id_map,
    "hpo_gene_evidence_mapped.csv.gz": hpo_gene_evidence,
    "clingen_gene_evidence_mapped.csv.gz": clingen_gene_evidence,
    "hpo_unmapped_genes.csv.gz": unmapped_hpo_genes
}

for filename, table in tables_to_save.items():
    destination = processed_folder / filename
    temporary = processed_folder / filename.replace(
        ".csv.gz", ".partial.csv.gz"
    )

    expected = table.reset_index(drop=True).astype("string")

    if destination.exists():
        file_to_check = destination
    else:
        expected.to_csv(temporary, index=False, compression="gzip")
        file_to_check = temporary

    saved = pd.read_csv(
        file_to_check,
        dtype="string",
        keep_default_na=False
    ).replace({"": pd.NA})

    pd.testing.assert_frame_equal(
        saved,
        expected.replace({"": pd.NA})
    )

    if file_to_check != destination:
        temporary.replace(destination)

    print(f"Verified: {filename}")

Verified: gene_id_map_clinvar_2026-09.csv.gz
Verified: hpo_gene_evidence_mapped.csv.gz
Verified: clingen_gene_evidence_mapped.csv.gz
Verified: hpo_unmapped_genes.csv.gz


In [56]:
import gzip
from collections import Counter

latest_snapshot = snapshots[-1]

source_file = (
    processed_folder / f"clinvar_variants_{latest_snapshot}.csv.gz"
)
destination = (
    processed_folder / f"clinvar_variant_gene_{latest_snapshot}.csv.gz"
)
temporary = (
    processed_folder / f"clinvar_variant_gene_{latest_snapshot}.partial.csv.gz"
)

# One HGNC partner per NCBI gene ID, as checked earlier.
hgnc_lookup = gene_id_map.set_index("ncbi_gene_id")["hgnc_id"]
assert hgnc_lookup.index.is_unique

output_columns = [
    "VariationID",
    "snapshot",
    "GeneID",
    "GeneSymbol",
    "HGNC_ID",
    "ncbi_gene_id",
    "mapped_hgnc_id",
    "link_status"
]

if not destination.exists():
    print("Building the current variant-to-gene lookup...")

    with gzip.open(temporary, "wt", encoding="utf-8", newline="") as handle:
        pd.DataFrame(columns=output_columns).to_csv(handle, index=False)

        for chunk in pd.read_csv(
            source_file,
            usecols=["VariationID", "GeneID", "GeneSymbol", "HGNC_ID"],
            dtype="string",
            keep_default_na=False,
            chunksize=100_000
        ):
            gene_ids = chunk["GeneID"].str.strip()

            valid_gene = gene_ids.str.fullmatch(
                r"[1-9][0-9]*", na=False
            )

            # Keep the original fields and add normalized linking fields.
            chunk["snapshot"] = latest_snapshot
            chunk["ncbi_gene_id"] = gene_ids.where(valid_gene, "")
            chunk["mapped_hgnc_id"] = (
                chunk["ncbi_gene_id"].map(hgnc_lookup).fillna("")
            )

            chunk["link_status"] = "unresolved_gene_id"
            chunk.loc[valid_gene, "link_status"] = "linked_ncbi_only"
            chunk.loc[
                valid_gene & chunk["mapped_hgnc_id"].ne(""),
                "link_status"
            ] = "linked_ncbi_and_hgnc"

            chunk[output_columns].to_csv(
                handle, index=False, header=False
            )

    file_to_check = temporary
else:
    print("Lookup already exists; checking it.")
    file_to_check = destination

# Check the lookup against the source, one chunk at a time.
lookup_chunks = pd.read_csv(
    file_to_check,
    dtype="string",
    keep_default_na=False,
    chunksize=100_000
)

source_chunks = pd.read_csv(
    source_file,
    usecols=["VariationID", "GeneID", "GeneSymbol", "HGNC_ID"],
    dtype="string",
    keep_default_na=False,
    chunksize=100_000
)

from itertools import zip_longest

status_counts = Counter()
checked_rows = 0
unresolved_examples = []

for saved, original in zip_longest(lookup_chunks, source_chunks):
    assert saved is not None and original is not None, "File lengths differ."
    assert list(saved.columns) == output_columns

    for column in ["VariationID", "GeneID", "GeneSymbol", "HGNC_ID"]:
        assert saved[column].equals(original[column]), (
            f"Source mismatch in {column}."
        )

    assert saved["snapshot"].eq(latest_snapshot).all()

    gene_ids = original["GeneID"].str.strip()
    valid_gene = gene_ids.str.fullmatch(r"[1-9][0-9]*", na=False)
    expected_ncbi = gene_ids.where(valid_gene, "")
    expected_hgnc = expected_ncbi.map(hgnc_lookup).fillna("")

    assert saved["ncbi_gene_id"].eq(expected_ncbi).all()
    assert saved["mapped_hgnc_id"].eq(expected_hgnc).all()

    expected_status = pd.Series(
        "unresolved_gene_id", index=saved.index, dtype="string"
    )
    expected_status.loc[valid_gene] = "linked_ncbi_only"
    expected_status.loc[valid_gene & expected_hgnc.ne("")] = "linked_ncbi_and_hgnc"

    assert saved["link_status"].eq(expected_status).all()

    if not unresolved_examples:
        examples = saved.loc[~valid_gene].head(10)
        if not examples.empty:
            unresolved_examples.append(examples)

    checked_rows += len(saved)
    status_counts.update(saved["link_status"])

if file_to_check != destination:
    temporary.replace(destination)

print("\nVARIANT–GENE LOOKUP VERIFIED")
print(f"Variant records: {checked_rows:,}")

for status, count in status_counts.items():
    print(f"{status}: {count:,}")

if unresolved_examples:
    print("\nUnresolved examples:")
    display(unresolved_examples[0])

print(f"\nSaved to: {destination}")

Building the current variant-to-gene lookup...

VARIANT–GENE LOOKUP VERIFIED
Variant records: 4,243,470
linked_ncbi_and_hgnc: 4,225,515
unresolved_gene_id: 17,635
linked_ncbi_only: 320

Unresolved examples:


,VariationID,snapshot,GeneID,GeneSymbol,HGNC_ID,ncbi_gene_id,mapped_hgnc_id,link_status
184,192,2026-09,-1,ATXN8;ATXN8OS;LOC109461478,-,,,unresolved_gene_id
912,973,2026-09,-1,FANCI;POLG,-,,,unresolved_gene_id
1070,1143,2026-09,-1,DDX25;HYLS1;PUS3,-,,,unresolved_gene_id
1585,1716,2026-09,-1,ERCC8;NDUFAF2,-,,,unresolved_gene_id
1804,1966,2026-09,-1,ADA;PKIG,-,,,unresolved_gene_id
1890,2062,2026-09,-1,SUMO4;TAB2,-,,,unresolved_gene_id
1915,2087,2026-09,-1,GCDH;KLF1;LOC117125594,-,,,unresolved_gene_id
1944,2120,2026-09,-1,LLGL2;TSEN54,-,,,unresolved_gene_id
2315,2527,2026-09,-1,ASPN;CENPP,-,,,unresolved_gene_id
2358,2573,2026-09,-1,APOC4-APOC2;APOC2,-,,,unresolved_gene_id



Saved to: D:\GeneVISTA\data\processed\clinvar_variant_gene_2026-09.csv.gz


In [57]:
# Count prediction outcomes before building features or training.
split_by_start = {
    "2022-01": "train",
    "2023-01": "train",
    "2024-01": "validation",
    "2025-01": "test",
    "2026-01": "short_interval_audit"
}

outcome_rows = []

for earlier, later in zip(snapshots[:-1], snapshots[1:]):
    starts_as_vus = timeline[earlier].eq("vus")
    outcomes = timeline.loc[starts_as_vus, later].value_counts()

    stayed_vus = int(outcomes.get("vus", 0))
    became_benign = int(outcomes.get("benign", 0))
    became_pathogenic = int(outcomes.get("pathogenic", 0))

    became_conflicting = int(outcomes.get("conflicting", 0))
    became_other = int(outcomes.get("other", 0))
    excluded_later = int(outcomes.get("present_excluded", 0))
    absent_later = int(outcomes.get("absent_grch38", 0))

    eligible_rows = stayed_vus + became_benign + became_pathogenic
    total_vus = int(starts_as_vus.sum())

    assert (
        eligible_rows
        + became_conflicting
        + became_other
        + excluded_later
        + absent_later
    ) == total_vus

    outcome_rows.append({
        "split": split_by_start[earlier],
        "from_snapshot": earlier,
        "to_snapshot": later,
        "starting_vus": total_vus,
        "stayed_vus": stayed_vus,
        "became_benign": became_benign,
        "became_pathogenic": became_pathogenic,
        "became_conflicting": became_conflicting,
        "became_other": became_other,
        "excluded_later": excluded_later,
        "absent_later": absent_later,
        "eligible_for_three_class_model": eligible_rows,
        "eligible_percent": (
            round(100 * eligible_rows / total_vus, 2)
            if total_vus else None
        )
    })

vus_outcome_summary = pd.DataFrame(outcome_rows)

print("VUS OUTCOME COUNTS")
print(vus_outcome_summary.to_string(index=False))

print("\nCLASS COUNTS BY MODEL SPLIT")
split_counts = (
    vus_outcome_summary.loc[
        vus_outcome_summary["split"].ne("short_interval_audit")
    ]
    .groupby("split")[[
        "stayed_vus",
        "became_benign",
        "became_pathogenic"
    ]]
    .sum()
    .reindex(["train", "validation", "test"])
)

print(split_counts.to_string())

VUS OUTCOME COUNTS
               split from_snapshot to_snapshot  starting_vus  stayed_vus  became_benign  became_pathogenic  became_conflicting  became_other  excluded_later  absent_later  eligible_for_three_class_model  eligible_percent
               train       2022-01     2023-01        421102      404842           1338               1135               12904           125               1           757                          407315             96.73
               train       2023-01     2024-01        641852      603237           9906               2049               25695            83               0           882                          615192             95.85
          validation       2024-01     2025-01       1138685     1113838           6325               1769               16693            32               1            27                         1121932             98.53
                test       2025-01     2026-01       1475196     1447648           7098          

In [58]:
label_folder = processed_folder / "model_labels"
label_folder.mkdir(parents=True, exist_ok=True)

target_names = {
    "vus": "stayed_vus",
    "benign": "became_benign",
    "pathogenic": "became_pathogenic"
}

model_intervals = [
    ("2022-01", "2023-01", "train"),
    ("2023-01", "2024-01", "train"),
    ("2024-01", "2025-01", "validation"),
    ("2025-01", "2026-01", "test")
]

label_manifest = []

for earlier, later, split in model_intervals:
    eligible = (
        timeline[earlier].eq("vus")
        & timeline[later].isin(target_names)
    )

    # Each example represents a variant at a particular starting snapshot.
    labels = pd.DataFrame({
        "VariationID": timeline.index[eligible].to_numpy(),
        "feature_snapshot": earlier,
        "outcome_snapshot": later,
        "split": split,
        "target": (
            timeline.loc[eligible, later]
            .astype("string")
            .map(target_names)
            .to_numpy()
        )
    })

    assert not labels.empty
    assert not labels.duplicated(
        ["VariationID", "feature_snapshot"]
    ).any()
    assert labels["target"].isin(target_names.values()).all()

    expected_count = int(
        vus_outcome_summary.loc[
            vus_outcome_summary["from_snapshot"].eq(earlier),
            "eligible_for_three_class_model"
        ].iloc[0]
    )
    assert len(labels) == expected_count

    filename = f"vus_labels_{earlier}_to_{later}.csv.gz"
    destination = label_folder / filename
    temporary = label_folder / filename.replace(
        ".csv.gz", ".partial.csv.gz"
    )

    if destination.exists():
        file_to_check = destination
    else:
        labels.to_csv(
            temporary,
            index=False,
            compression="gzip",
            chunksize=100_000
        )
        file_to_check = temporary

    # Compare every saved row with the labels just constructed.
    checked_rows = 0

    for saved in pd.read_csv(
        file_to_check,
        dtype="string",
        keep_default_na=False,
        chunksize=100_000
    ):
        expected = labels.iloc[
            checked_rows:checked_rows + len(saved)
        ].astype("string")

        pd.testing.assert_frame_equal(
            saved.reset_index(drop=True),
            expected.reset_index(drop=True)
        )

        checked_rows += len(saved)

    assert checked_rows == len(labels)

    if file_to_check != destination:
        temporary.replace(destination)

    counts = labels["target"].value_counts()

    label_manifest.append({
        "file": filename,
        "split": split,
        "feature_snapshot": earlier,
        "outcome_snapshot": later,
        "rows": len(labels),
        **{
            target: int(counts.get(target, 0))
            for target in target_names.values()
        }
    })

    print(f"Verified: {filename} — {len(labels):,} examples")

label_manifest = pd.DataFrame(label_manifest)

display(label_manifest)

Verified: vus_labels_2022-01_to_2023-01.csv.gz — 407,315 examples
Verified: vus_labels_2023-01_to_2024-01.csv.gz — 615,192 examples
Verified: vus_labels_2024-01_to_2025-01.csv.gz — 1,121,932 examples
Verified: vus_labels_2025-01_to_2026-01.csv.gz — 1,456,851 examples


,file,split,feature_snapshot,outcome_snapshot,rows,stayed_vus,became_benign,became_pathogenic
0,vus_labels_2022-01_to_2023-01.csv.gz,train,2022-01,2023-01,407315,404842,1338,1135
1,vus_labels_2023-01_to_2024-01.csv.gz,train,2023-01,2024-01,615192,603237,9906,2049
2,vus_labels_2024-01_to_2025-01.csv.gz,validation,2024-01,2025-01,1121932,1113838,6325,1769
3,vus_labels_2025-01_to_2026-01.csv.gz,test,2025-01,2026-01,1456851,1447648,7098,2105


In [59]:
import json

manifest_file = label_folder / "label_manifest.csv"

if manifest_file.exists():
    saved_manifest = pd.read_csv(manifest_file, dtype="string")

    pd.testing.assert_frame_equal(
        saved_manifest,
        label_manifest.reset_index(drop=True).astype("string")
    )
else:
    label_manifest.to_csv(manifest_file, index=False)

preparation_notes = {
    "task": "Predict next annual snapshot state for variants starting as VUS",
    "targets": [
        "stayed_vus",
        "became_benign",
        "became_pathogenic"
    ],
    "train_intervals": ["2022-01_to_2023-01", "2023-01_to_2024-01"],
    "validation_interval": "2024-01_to_2025-01",
    "test_interval": "2025-01_to_2026-01",
    "excluded_target_outcomes": [
        "conflicting", "other", "present_excluded", "absent_grch38"
    ],
    "feature_rules": [
        "Use only the starting snapshot or earlier observations.",
        "Do not use VariationID as a model feature.",
        "Do not use future outcome fields as features.",
        "Fit imputers, encoders, and scalers on training data only.",
        "Keep September 2026 supporting evidence out of historical features.",
        "Reserve the test interval for final evaluation."
    ],
    "evaluation_scope": (
        "Future outcomes among eligible observed variants; "
        "variants may recur across chronological splits."
    ),
    "remaining_integration_limits": [
        "31 HPO genes lack an HGNC mapping in the current mapping.",
        "Unresolved variant-to-gene records are retained for review.",
        "MONDO and OMIM/ORPHA disease identities are not yet cross-mapped."
    ]
}

notes_file = label_folder / "preparation_notes.json"

if notes_file.exists():
    saved_notes = json.loads(notes_file.read_text(encoding="utf-8"))
    assert saved_notes == preparation_notes, "Existing notes differ."
else:
    notes_file.write_text(
        json.dumps(preparation_notes, indent=2),
        encoding="utf-8"
    )

print("PREPARATION CHECKPOINT SAVED")
print(f"Manifest: {manifest_file}")
print(f"Notes: {notes_file}")

PREPARATION CHECKPOINT SAVED
Manifest: D:\GeneVISTA\data\processed\model_labels\label_manifest.csv
Notes: D:\GeneVISTA\data\processed\model_labels\preparation_notes.json
